In [ ]:
!pip install -q deep-translator sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.7 MB/s eta 0:00:00


In [ ]:
from transformers import MBartForConditionalGeneration, MBart50Tokenizer
from deep_translator import GoogleTranslator
import sacrebleu, torch, pandas as pd, time, os
from tqdm import tqdm

# ============================================================
# CONFIG — English → Marathi
# ============================================================
OUR_SRC_LANG = "en_XX"
OUR_TGT_LANG = "mr_IN"
GOOGLE_SRC   = "en"
GOOGLE_TGT   = "mr"        # ✅ Google uses "mr" NOT "mar"

MODEL_PATH   = "/content/drive/MyDrive/resources/translation/model/English_Marathi"
TEST_CSV     = "/content/drive/MyDrive/test-mar-eng.csv"
SRC_COL      = "english"   # ✅ lowercase (matches CSV column)
TGT_COL      = "marathi"   # ✅ lowercase (matches CSV column)
SAMPLE_SIZE  = 100
# ============================================================

# 1. VERIFY MODEL PATH
assert os.path.exists(MODEL_PATH), f"❌ Model not found at: {MODEL_PATH}"
print(f"✅ Model path verified. Files: {os.listdir(MODEL_PATH)}")

# 2. LOAD TOKENIZER + MODEL
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

print("⏳ Loading tokenizer...")
tokenizer = MBart50Tokenizer.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt",
    src_lang=OUR_SRC_LANG,
    tgt_lang=OUR_TGT_LANG
)
print("✅ Tokenizer loaded")

print("⏳ Loading fine-tuned model...")
finetuned_model = MBartForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    local_files_only=True
).to(device)
finetuned_model.eval()
print("✅ Model loaded\n")

# 3. LOAD TEST DATA
df = pd.read_csv(TEST_CSV)
print(f"📋 CSV columns: {list(df.columns)}")

df = df[[SRC_COL, TGT_COL]].dropna()
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"📂 Loaded {len(sample_df)} test samples")
print(f"🔄 Comparing: English → Marathi\n")
print("-" * 60)

# 4. GOOGLE TRANSLATE
google_translations = []
print("🌐 Getting Google Translate outputs...")
gt = GoogleTranslator(source=GOOGLE_SRC, target=GOOGLE_TGT)

for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Google Translate"):
    try:
        result = gt.translate(row[SRC_COL])
        google_translations.append(result if result else "")
        time.sleep(0.05)
    except Exception as e:
        google_translations.append("")
        print(f"  ⚠️ Google error: {e}")

# 5. OUR MODEL
our_translations = []
print("\n🤖 Getting Our Fine-tuned Model outputs...")
tokenizer.src_lang = OUR_SRC_LANG

for i in tqdm(range(0, len(sample_df), 8), desc="Our Model"):
    batch     = sample_df.iloc[i:i+8]
    src_texts = batch[SRC_COL].tolist()

    inputs = tokenizer(
        src_texts, return_tensors="pt",
        padding=True, truncation=True, max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = finetuned_model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id[OUR_TGT_LANG],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    our_translations.extend(preds)

# 6. COMPUTE BLEU
references  = sample_df[TGT_COL].tolist()
bleu_google = sacrebleu.corpus_bleu(google_translations, [references]).score
bleu_ours   = sacrebleu.corpus_bleu(our_translations,   [references]).score

# 7. RESULTS SUMMARY
winner = "🌐 Google" if bleu_google > bleu_ours else "🤖 Our Model"
diff   = abs(bleu_google - bleu_ours)

print("\n" + "="*60)
print("       📊  TRANSLATION COMPARISON RESULTS")
print("="*60)
print(f"  Language     : English → Marathi")
print(f"  Test Samples : {SAMPLE_SIZE}")
print("-"*60)
print(f"  🌐 Google Translate BLEU : {bleu_google:.2f}")
print(f"  🤖 Our Fine-tuned BLEU   : {bleu_ours:.2f}")
print(f"  🏆 Winner                : {winner} (by {diff:.2f} points)")
print("="*60)

# 8. SIDE-BY-SIDE TABLE
results_df = pd.DataFrame({
    "Source (English)" : sample_df[SRC_COL].values,
    "Reference (Marathi)" : references,
    "Google"           : google_translations,
    "Our Model"        : our_translations
})

print("\n📋 Side-by-Side Sample (First 10 rows):")
pd.set_option('display.max_colwidth', 55)
print(results_df.head(10).to_string(index=False))

# 9. SAVE TO DRIVE
RESULTS_PATH = "/content/drive/MyDrive/comparison_en_mr.csv"
results_df.to_csv(RESULTS_PATH, index=False)
print(f"\n💾 Full results saved to: {RESULTS_PATH}")

✅ Model path verified. Files: ['config.json', 'generation_config.json', 'model.safetensors']
🖥️  Device: cuda
⏳ Loading tokenizer...
✅ Tokenizer loaded
⏳ Loading fine-tuned model...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

✅ Model loaded

📋 CSV columns: ['english', 'marathi']
📂 Loaded 100 test samples
🔄 Comparing: English → Marathi

------------------------------------------------------------
🌐 Getting Google Translate outputs...


Google Translate: 100%|██████████| 100/100 [00:46<00:00,  2.14it/s]



🤖 Getting Our Fine-tuned Model outputs...


Our Model: 100%|██████████| 13/13 [00:29<00:00,  2.27s/it]


       📊  TRANSLATION COMPARISON RESULTS
  Language     : English → Marathi
  Test Samples : 100
------------------------------------------------------------
  🌐 Google Translate BLEU : 17.57
  🤖 Our Fine-tuned BLEU   : 3.84
  🏆 Winner                : 🌐 Google (by 13.73 points)

📋 Side-by-Side Sample (First 10 rows):
                                                                                                                                                             Source (English)                                                                                                                                Reference (Marathi)                                                                                                                                                    Google                                                                                                                                                     Our Model
                                              

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50Tokenizer
from deep_translator import GoogleTranslator
import sacrebleu, torch, pandas as pd, time, os
from tqdm import tqdm

OUR_SRC_LANG = "en_XX"
OUR_TGT_LANG = "hi_IN"
GOOGLE_SRC   = "en"
GOOGLE_TGT   = "hi"
MODEL_PATH   = "/content/drive/MyDrive/resources/translation/model/English_Hindi"

# ✏️ Update these if your CSV path/columns are different
TEST_CSV     = "/content/drive/MyDrive/test-eng-hi.csv"
SRC_COL      = "english"   # ← Update if needed after seeing CSV columns below
TGT_COL      = "hindi"
SAMPLE_SIZE  = 100

assert os.path.exists(MODEL_PATH), f"❌ Not found: {MODEL_PATH}"
print(f"✅ Model files: {os.listdir(MODEL_PATH)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt", src_lang=OUR_SRC_LANG, tgt_lang=OUR_TGT_LANG)
finetuned_model = MBartForConditionalGeneration.from_pretrained(MODEL_PATH, local_files_only=True).to(device)
finetuned_model.eval()
print("✅ Model ready\n")

# Load CSV — prints columns so you can verify
df = pd.read_csv(TEST_CSV)
print(f"📋 CSV columns: {list(df.columns)}")
df = df[[SRC_COL, TGT_COL]].dropna()
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"📂 Loaded {len(sample_df)} samples\n")

google_translations = []
gt = GoogleTranslator(source=GOOGLE_SRC, target=GOOGLE_TGT)
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="🌐 Google"):
    try:
        result = gt.translate(row[SRC_COL])
        google_translations.append(result if result else "")
        time.sleep(0.05)
    except:
        google_translations.append("")

our_translations = []
tokenizer.src_lang = OUR_SRC_LANG
for i in tqdm(range(0, len(sample_df), 8), desc="🤖 Our Model"):
    batch = sample_df.iloc[i:i+8]
    inputs = tokenizer(batch[SRC_COL].tolist(), return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = finetuned_model.generate(**inputs, forced_bos_token_id=tokenizer.lang_code_to_id[OUR_TGT_LANG], max_length=128, num_beams=4, early_stopping=True)
    our_translations.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))

references  = sample_df[TGT_COL].tolist()
bleu_google = sacrebleu.corpus_bleu(google_translations, [references]).score
bleu_ours   = sacrebleu.corpus_bleu(our_translations,   [references]).score
winner = "🌐 Google" if bleu_google > bleu_ours else "🤖 Our Model"

print("\n" + "="*60)
print("     📊 COMPARISON: English → Hindi")
print("="*60)
print(f"  🌐 Google Translate BLEU : {bleu_google:.2f}")
print(f"  🤖 Our Fine-tuned BLEU   : {bleu_ours:.2f}")
print(f"  🏆 Winner                : {winner} (by {abs(bleu_google - bleu_ours):.2f} pts)")
print("="*60)

results_df = pd.DataFrame({"Source (English)": sample_df[SRC_COL].values, "Reference (Hindi)": references, "Google": google_translations, "Our Model": our_translations})
pd.set_option('display.max_colwidth', 55)
print(results_df.head(10).to_string(index=False))
results_df.to_csv("/content/drive/MyDrive/comparison_en_hi.csv", index=False)
print("\n💾 Saved to Drive: comparison_en_hi.csv")

✅ Model files: ['config.json', 'generation_config.json', 'model.safetensors']
🖥️  Device: cuda


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

✅ Model ready

📋 CSV columns: ['hindi', 'english']
📂 Loaded 100 samples



🤖 Our Model: 100%|██████████| 13/13 [00:22<00:00,  1.74s/it]


     📊 COMPARISON: English → Hindi
  🌐 Google Translate BLEU : 29.11
  🤖 Our Fine-tuned BLEU   : 23.39
  🏆 Winner                : 🌐 Google (by 5.72 pts)
                                                                                                                                                                    Source (English)                                                                                                                                                                  Reference (Hindi)                                                                                                                                                                          Google                                                                                                                                                                     Our Model
                                                                                        The vast majority of new cases were transmitted

In [ ]:
from transformers import MBartForConditionalGeneration, MBart50Tokenizer
from deep_translator import GoogleTranslator
import sacrebleu, torch, pandas as pd, time, os
from tqdm import tqdm

OUR_SRC_LANG = "hi_IN"
OUR_TGT_LANG = "en_XX"
GOOGLE_SRC   = "hi"
GOOGLE_TGT   = "en"
MODEL_PATH   = "/content/drive/MyDrive/resources/translation/model/Hindi_English"

# ✏️ Update these if your CSV path/columns are different
TEST_CSV     = "/content/drive/MyDrive/test-eng-hi.csv"
SRC_COL      = "hindi"
TGT_COL      = "english"   # ← Update if needed after seeing CSV columns below
SAMPLE_SIZE  = 100

assert os.path.exists(MODEL_PATH), f"❌ Not found: {MODEL_PATH}"
print(f"✅ Model files: {os.listdir(MODEL_PATH)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Device: {device}")

tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt", src_lang=OUR_SRC_LANG, tgt_lang=OUR_TGT_LANG)
finetuned_model = MBartForConditionalGeneration.from_pretrained(MODEL_PATH, local_files_only=True).to(device)
finetuned_model.eval()
print("✅ Model ready\n")

# Load CSV — prints columns so you can verify
df = pd.read_csv(TEST_CSV)
print(f"📋 CSV columns: {list(df.columns)}")
df = df[[SRC_COL, TGT_COL]].dropna()
sample_df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"📂 Loaded {len(sample_df)} samples\n")

google_translations = []
gt = GoogleTranslator(source=GOOGLE_SRC, target=GOOGLE_TGT)
for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="🌐 Google"):
    try:
        result = gt.translate(row[SRC_COL])
        google_translations.append(result if result else "")
        time.sleep(0.05)
    except:
        google_translations.append("")

our_translations = []
tokenizer.src_lang = OUR_SRC_LANG
for i in tqdm(range(0, len(sample_df), 8), desc="🤖 Our Model"):
    batch = sample_df.iloc[i:i+8]
    inputs = tokenizer(batch[SRC_COL].tolist(), return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = finetuned_model.generate(**inputs, forced_bos_token_id=tokenizer.lang_code_to_id[OUR_TGT_LANG], max_length=128, num_beams=4, early_stopping=True)
    our_translations.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))

references  = sample_df[TGT_COL].tolist()
bleu_google = sacrebleu.corpus_bleu(google_translations, [references]).score
bleu_ours   = sacrebleu.corpus_bleu(our_translations,   [references]).score
winner = "🌐 Google" if bleu_google > bleu_ours else "🤖 Our Model"

print("\n" + "="*60)
print("     📊 COMPARISON: Hindi → English")
print("="*60)
print(f"  🌐 Google Translate BLEU : {bleu_google:.2f}")
print(f"  🤖 Our Fine-tuned BLEU   : {bleu_ours:.2f}")
print(f"  🏆 Winner                : {winner} (by {abs(bleu_google - bleu_ours):.2f} pts)")
print("="*60)

results_df = pd.DataFrame({"Source (Hindi)": sample_df[SRC_COL].values, "Reference (English)": references, "Google": google_translations, "Our Model": our_translations})
pd.set_option('display.max_colwidth', 55)
print(results_df.head(10).to_string(index=False))
results_df.to_csv("/content/drive/MyDrive/comparison_hi_en.csv", index=False)
print("\n💾 Saved to Drive: comparison_hi_en.csv")

✅ Model files: ['config.json', 'generation_config.json', 'model.safetensors']
🖥️  Device: cuda


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

✅ Model ready

📋 CSV columns: ['hindi', 'english']
📂 Loaded 100 samples



🤖 Our Model: 100%|██████████| 13/13 [00:22<00:00,  1.70s/it]


     📊 COMPARISON: Hindi → English
  🌐 Google Translate BLEU : 42.64
  🤖 Our Fine-tuned BLEU   : 30.42
  🏆 Winner                : 🌐 Google (by 12.22 pts)
                                                                                                                                                                    Source (Hindi)                                                                                                                                                                  Reference (English)                                                                                                                                                         Google                                                                                                                                                                        Our Model
                                                                                       पिछले कुछ समय में हुए बदलावों को देखते हुए, ज़्यादातर नए मामलो

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


ENGLISH TO MARATHI INDIC VS GOOGLW

In [ ]:
!pip install -q IndicTransToolkit
!pip install -q deep-translator sacrebleu tabulate


In [ ]:
# """
# ╔══════════════════════════════════════════════════════════════════╗
# ║   English → Marathi: IndicTrans2  vs  Google Translate           ║
# ║   ✅ NO IndicTransToolkit dependency — works on any Colab        ║
# ╠══════════════════════════════════════════════════════════════════╣
# ║  STEP 1  →  Run the install cell below FIRST                     ║
# ║  STEP 2  →  Run cells in order (each ━━ block = one Colab cell)  ║
# ╚══════════════════════════════════════════════════════════════════╝
# """

# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 1  ── INSTALL  (run this first, ~30 sec)
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # Paste and run in Colab:
# #
# #   !pip install -q deep-translator sacrebleu tabulate
# #   !pip install -q sentencepiece
# #
# # That's it — NO IndicTransToolkit needed anymore ✅


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 2  ── IMPORTS & CONFIG
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# import torch
# import time
# import warnings
# import unicodedata
# warnings.filterwarnings("ignore")

# from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# # ── Optional: Google Translate ────────────────────────────────────
# try:
#     from deep_translator import GoogleTranslator
#     HAS_GOOGLE = True
#     print("✅ deep_translator   : Ready")
# except ImportError:
#     HAS_GOOGLE = False
#     print("⚠️  deep_translator missing  →  !pip install deep-translator")

# # ── Optional: BLEU scoring ────────────────────────────────────────
# try:
#     import sacrebleu
#     HAS_BLEU = True
#     print("✅ sacrebleu         : Ready")
# except ImportError:
#     HAS_BLEU = False

# # ── Optional: Pretty tables ───────────────────────────────────────
# try:
#     from tabulate import tabulate
#     HAS_TABULATE = True
#     print("✅ tabulate          : Ready")
# except ImportError:
#     HAS_TABULATE = False

# # ── Device ────────────────────────────────────────────────────────
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"\n🖥️  Device        : {DEVICE.upper()}")

# # ── Model choice ──────────────────────────────────────────────────
# #   200M  → fast,  ~400 MB download,  good accuracy
# #   1B    → slow,  ~2 GB  download,   BEST  (closest to beating Google)
# MODEL_NAME = "ai4bharat/indictrans2-en-indic-dist-200M"
# # MODEL_NAME = "ai4bharat/indictrans2-en-indic-1B"   # ← uncomment for max accuracy

# SRC_LANG = "eng_Latn"   # English  (latin script)
# TGT_LANG = "mar_Deva"   # Marathi  (Devanagari script)

# print(f"📦 Model          : {MODEL_NAME}")
# print(f"🔁 Direction      : {SRC_LANG}  →  {TGT_LANG}  (English → Marathi)")


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 3  ── LOAD MODEL  (downloads once, ~400 MB for 200M variant)
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# print("\n⏳ Loading tokenizer & model — first run downloads from HuggingFace...")
# _t0 = time.time()

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_NAME,
#     trust_remote_code=True,
# )

# model = AutoModelForSeq2SeqLM.from_pretrained(
#     MODEL_NAME,
#     trust_remote_code=True,
#     # float16 on GPU (faster), float32 on CPU (stable)
#     torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
#     low_cpu_mem_usage=True,
# )
# model.to(DEVICE)
# model.eval()

# _load_time = time.time() - _t0
# print(f"✅ Model loaded in {_load_time:.1f}s\n")

# # ── Resolve target language token id ─────────────────────────────
# # IndicTrans2 tokenizer has Flores-200 language codes as special tokens.
# # We use convert_tokens_to_ids instead of IndicTransToolkit.
# _tgt_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

# # Sanity check
# if _tgt_token_id == tokenizer.unk_token_id:
#     # Fallback: try with angle brackets (some tokenizer versions use this format)
#     _tgt_token_id = tokenizer.convert_tokens_to_ids(f">>{TGT_LANG}<<")

# print(f"🎯 Target lang token  : '{TGT_LANG}'  →  id = {_tgt_token_id}")

# if _tgt_token_id == tokenizer.unk_token_id or _tgt_token_id is None:
#     print("⚠️  WARNING: Could not find Marathi token id in tokenizer!")
#     print("   The model may still work but with reduced quality.")
#     print("   Try the 1B model or verify the model name.")
# else:
#     print("✅ Language token resolved correctly!\n")


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 4  ── TRANSLATION FUNCTIONS
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# def _preprocess_english(sentences: list[str]) -> list[str]:
#     """
#     Lightweight English preprocessing — replaces IndicProcessor for English input.
#     IndicTransToolkit's preprocess_batch does heavy normalization for Indic scripts.
#     For English (Latin script) input, basic cleaning is sufficient.
#     """
#     cleaned = []
#     for s in sentences:
#         s = s.strip()
#         # Normalize unicode (NFC form — consistent character composition)
#         s = unicodedata.normalize("NFC", s)
#         # Collapse multiple spaces
#         s = " ".join(s.split())
#         cleaned.append(s)
#     return cleaned


# def translate_indictrans2(sentences: list[str]) -> tuple[list[str], float]:
#     """
#     Translate a list of English sentences → Marathi using IndicTrans2.
#     No IndicTransToolkit needed.
#     Returns: (translations, elapsed_seconds)
#     """
#     t0 = time.time()

#     # Preprocess
#     batch = _preprocess_english(sentences)

#     # Set source language on the tokenizer
#     tokenizer.src_lang = SRC_LANG

#     # Tokenize
#     inputs = tokenizer(
#         batch,
#         truncation=True,
#         padding="longest",
#         return_tensors="pt",
#         return_attention_mask=True,
#         max_length=256,
#     ).to(DEVICE)

#     # Generate
#     with torch.no_grad():
#         generated_tokens = model.generate(
#             **inputs,
#             forced_bos_token_id=_tgt_token_id,   # Force Marathi output
#             use_cache=True,
#             min_length=0,
#             max_length=256,
#             num_beams=5,            # Beam search = higher quality
#             num_return_sequences=1,
#             early_stopping=True,
#         )

#     # Decode
#     translations = tokenizer.batch_decode(
#         generated_tokens,
#         skip_special_tokens=True,
#         clean_up_tokenization_spaces=True,
#     )

#     return translations, time.time() - t0


# def translate_google(sentences: list[str]) -> tuple[list[str], float]:
#     """
#     Translate English → Marathi via Google Translate (free, unofficial).
#     Returns: (translations, elapsed_seconds)
#     """
#     if not HAS_GOOGLE:
#         return ["[Install: !pip install deep-translator]"] * len(sentences), 0.0

#     gt  = GoogleTranslator(source="en", target="mr")
#     out = []
#     t0  = time.time()

#     for sent in sentences:
#         try:
#             res = gt.translate(sent)
#             out.append(res or "[Empty]")
#         except Exception as e:
#             out.append(f"[Err: {str(e)[:35]}]")
#         time.sleep(0.2)   # Gentle rate-limit

#     return out, time.time() - t0


# print("✅ Functions ready!\n")


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 5  ── TEST SENTENCES  (edit freely!)
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# test_sentences = [
#     # ── Simple / everyday ─────────────────────────────────────
#     "Hello, how are you?",
#     "What is your name?",
#     "I am going to the market tomorrow.",
#     "The weather is very pleasant today.",
#     "Please help me with this work.",

#     # ── Medium complexity ─────────────────────────────────────
#     "He studies at Mumbai University.",
#     "India is a democratic country.",
#     "The train arrives at five o'clock in the evening.",
#     "We should respect our elders.",
#     "Technology has changed our lives completely.",

#     # ── Complex / domain-specific ─────────────────────────────
#     "The government has implemented a new policy for education.",
#     "Farmers work very hard throughout the year.",
#     "Children should be given quality education from an early age.",
#     "The hospital is located near the main railway station.",
#     "She won the first prize in the national science competition.",
# ]

# print(f"📋 {len(test_sentences)} test sentences loaded.")
# print("   Run CELL 6 to start the comparison!\n")


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 6  ── RUN COMPARISON  🚀
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# print("=" * 72)
# print("  🚀  Running Translations...")
# print("=" * 72)

# # ── IndicTrans2 ──────────────────────────────────────────────────
# print("\n🔵  [1/2] IndicTrans2...")
# it2_out, it2_time = translate_indictrans2(test_sentences)
# print(f"     ✅ Done  {it2_time:.2f}s  ({it2_time/len(test_sentences):.2f}s/sent)")

# # ── Google Translate ─────────────────────────────────────────────
# print("\n🔴  [2/2] Google Translate...")
# if HAS_GOOGLE:
#     goo_out, goo_time = translate_google(test_sentences)
#     print(f"     ✅ Done  {goo_time:.2f}s  ({goo_time/len(test_sentences):.2f}s/sent)")
# else:
#     goo_out  = ["[pip install deep-translator]"] * len(test_sentences)
#     goo_time = 0.0
#     print("     ⚠️  Skipped — deep_translator not installed")


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 7  ── DISPLAY RESULTS
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# print("\n")
# print("═" * 90)
# print("   📊  SIDE-BY-SIDE  |  English → Marathi")
# print("═" * 90)

# for idx, (src, it2, goo) in enumerate(zip(test_sentences, it2_out, goo_out), 1):
#     print(f"\n  [{idx:02d}]  🇬🇧  {src}")
#     print(f"        🔵  IndicTrans2   :  {it2}")
#     print(f"        🔴  Google Trans  :  {goo}")
#     print("        " + "─" * 76)


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 8  ── BLEU SCORE  (add your reference translations below)
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# # ── Add your ground-truth Marathi translations here ───────────────
# # One Marathi string per English test sentence.
# # Leave as None to skip BLEU scoring.
# reference_translations = None

# # Example (uncomment + fill in):
# # reference_translations = [
# #     "नमस्कार, तुम्ही कसे आहात?",
# #     "तुमचे नाव काय आहे?",
# #     "मी उद्या बाजारात जाणार आहे.",
# #     "आज हवामान खूप आल्हाददायक आहे.",
# #     "कृपया माझ्या कामात मला मदत करा.",
# #     "तो मुंबई विद्यापीठात शिकतो.",
# #     "भारत एक लोकशाही देश आहे.",
# #     "संध्याकाळी पाच वाजता रेल्वे येते.",
# #     "आपण आपल्या वडिलांचा आदर केला पाहिजे.",
# #     "तंत्रज्ञानाने आपले जीवन पूर्णपणे बदलले आहे.",
# #     "सरकारने शिक्षणासाठी नवीन धोरण लागू केले आहे.",
# #     "शेतकरी वर्षभर खूप कष्ट करतात.",
# #     "मुलांना लहान वयातच चांगले शिक्षण दिले पाहिजे.",
# #     "रुग्णालय मुख्य रेल्वे स्थानकाजवळ आहे.",
# #     "तिने राष्ट्रीय विज्ञान स्पर्धेत प्रथम पुरस्कार जिंकला.",
# # ]

# if HAS_BLEU and reference_translations:
#     print("\n" + "═" * 70)
#     print("  📐  BLEU  SCORE  EVALUATION")
#     print("═" * 70)

#     it2_bleu = sacrebleu.corpus_bleu(it2_out, [reference_translations])
#     print(f"  🔵  IndicTrans2      BLEU : {it2_bleu.score:.2f}")

#     if HAS_GOOGLE and goo_time > 0:
#         goo_bleu = sacrebleu.corpus_bleu(goo_out, [reference_translations])
#         print(f"  🔴  Google Translate BLEU : {goo_bleu.score:.2f}")
#         winner = "🔵 IndicTrans2" if it2_bleu.score > goo_bleu.score else "🔴 Google Translate"
#         diff   = abs(it2_bleu.score - goo_bleu.score)
#         print(f"\n  🏆  Winner : {winner}  (+{diff:.2f} BLEU points ahead)")
# else:
#     print("\n  ℹ️   BLEU skipped — no reference translations provided.")
#     print("      Fill in `reference_translations` list in CELL 8 to enable.")


# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# # CELL 9  ── SUMMARY
# # ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# print("\n" + "═" * 70)
# print("  ⏱️   PERFORMANCE SUMMARY")
# print("═" * 70)
# print(f"  Sentences tested    :  {len(test_sentences)}")
# print(f"  Device              :  {DEVICE.upper()}")
# print(f"  Model               :  {MODEL_NAME.split('/')[-1]}")
# print(f"  Model load time     :  {_load_time:.1f}s")
# print(f"  🔵 IndicTrans2      :  {it2_time:.2f}s total  |  {it2_time/len(test_sentences):.2f}s/sent")
# if HAS_GOOGLE and goo_time > 0:
#     print(f"  🔴 Google Translate :  {goo_time:.2f}s total  |  {goo_time/len(test_sentences):.2f}s/sent  (+ network)")

# print("\n" + "═" * 70)
# print("  ✅  Comparison complete!")
# print("═" * 70)
# print("""
#   💡 Want even better accuracy?
#      ┌─────────────────────────────────────────────────────────┐
#      │  Switch to the 1B model in CELL 2:                      │
#      │  MODEL_NAME = "ai4bharat/indictrans2-en-indic-1B"       │
#      │                                                         │
#      │  Fine-tune on your 50k English-Marathi sentences        │
#      │  → will beat Google Translate by a large margin         │
#      └─────────────────────────────────────────────────────────┘
# """)


BULK TRANSALTION

In [ ]:
# ══════════════════════════════════════════════════════════════
#  SAFE BULK TRANSLATOR — Zero token cutoff, handles any text
# ══════════════════════════════════════════════════════════════

import torch, time, re, warnings
warnings.filterwarnings("ignore")
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

try:
    import nltk
    nltk.download('punkt_tab', quiet=True)
    from nltk.tokenize import sent_tokenize
    HAS_NLTK = True
except:
    HAS_NLTK = False

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
#MODEL_NAME = "facebook/nllb-200-distilled-600M"
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"

SRC_LANG   = "eng_Latn"
TGT_LANG   = "mar_Deva"
BATCH_SIZE = 16
NUM_BEAMS  = 4
TOKEN_LIMIT = 400   # Safe limit per chunk (NLLB max=1024, leave headroom)

print(f"🖥️  Device : {DEVICE.upper()}")

# ── Load model ────────────────────────────────────────────────
print("⏳ Loading model...")
_t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()
tokenizer.src_lang = SRC_LANG
_tgt_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f"✅ Model ready in {time.time()-_t0:.1f}s\n")


# ══════════════════════════════════════════════════════════════
#  STEP 1 — Count tokens BEFORE translating
# ══════════════════════════════════════════════════════════════

def count_tokens(text: str) -> int:
    """Count how many tokens a piece of text uses."""
    tokenizer.src_lang = SRC_LANG
    ids = tokenizer(text, add_special_tokens=True)["input_ids"]
    return len(ids)


# ══════════════════════════════════════════════════════════════
#  STEP 2 — Smart sub-splitting for oversized sentences
# ══════════════════════════════════════════════════════════════

# Priority order of split points (from least to most aggressive)
SPLIT_PATTERNS = [
    r'(?<=\.\s)',             # After period+space inside sentence
    r'(?<=;\s)',              # After semicolon
    r'(?<=,\s)',              # After comma
    r'\s+(?=and|but|or|however|therefore|moreover|furthermore|whereas)\s+',
    r'\s+',                  # Last resort: split at any space (by word count)
]

def sub_split_sentence(sentence: str, token_limit: int) -> list:
    """
    If a sentence exceeds token_limit, split it into safe sub-chunks
    at natural language boundaries WITHOUT cutting mid-word or mid-phrase.

    Strategy:
      1. Try splitting at semicolons first
      2. Then commas
      3. Then conjunctions
      4. Last resort: split by word count (hard limit)

    Each sub-chunk is guaranteed to fit within token_limit.
    """
    if count_tokens(sentence) <= token_limit:
        return [sentence]   # Fits fine — no splitting needed

    print(f"   ⚠️  Long sentence detected ({count_tokens(sentence)} tokens) — sub-splitting...")

    # Try each split pattern
    for pattern in SPLIT_PATTERNS:
        parts = re.split(pattern, sentence)
        parts = [p.strip() for p in parts if p.strip()]

        if len(parts) <= 1:
            continue  # This pattern didn't split it

        # Check if ALL parts fit within token limit
        # If a part is still too long, recursively split it
        safe_chunks = []
        all_fit = True

        for part in parts:
            if count_tokens(part) <= token_limit:
                safe_chunks.append(part)
            else:
                # Recursively split this part
                sub = sub_split_sentence(part, token_limit)
                safe_chunks.extend(sub)

        if safe_chunks:
            print(f"      → Split into {len(safe_chunks)} sub-chunks")
            return safe_chunks

    # Absolute last resort: split by words into equal halves
    words = sentence.split()
    mid   = len(words) // 2
    left  = ' '.join(words[:mid])
    right = ' '.join(words[mid:])
    print(f"      → Word-split fallback: {len(words)} words → {mid} + {len(words)-mid}")
    return sub_split_sentence(left, token_limit) + sub_split_sentence(right, token_limit)


# ══════════════════════════════════════════════════════════════
#  STEP 3 — Parse full document into safe translation units
# ══════════════════════════════════════════════════════════════

def parse_document(text: str) -> list:
    """
    Parse a multi-page document into a flat list of items:
      {'type': 'sentence', 'text': '...', 'tokens': N}
      {'type': 'para_break'}

    Every sentence is GUARANTEED to be within TOKEN_LIMIT.
    No text is ever discarded.
    """
    items       = []
    paragraphs  = re.split(r'\n\s*\n', text.strip())
    oversized   = 0

    for p_idx, para in enumerate(paragraphs):
        if p_idx > 0:
            items.append({'type': 'para_break'})

        para = para.strip()
        if not para:
            continue

        # Split paragraph into sentences
        raw_sentences = (sent_tokenize(para, language='english')
                         if HAS_NLTK else re.split(r'(?<=[.!?])\s+', para))

        for raw_sent in raw_sentences:
            raw_sent = raw_sent.strip()
            if not raw_sent:
                continue

            n_tokens = count_tokens(raw_sent)

            if n_tokens <= TOKEN_LIMIT:
                # ✅ Safe — translate as-is
                items.append({
                    'type'   : 'sentence',
                    'text'   : raw_sent,
                    'tokens' : n_tokens,
                })
            else:
                # ⚠️ Too long — sub-split at natural boundaries
                oversized += 1
                sub_chunks = sub_split_sentence(raw_sent, TOKEN_LIMIT)
                for chunk in sub_chunks:
                    items.append({
                        'type'   : 'sentence',
                        'text'   : chunk,
                        'tokens' : count_tokens(chunk),
                    })

    # Print token distribution summary
    sentence_items = [it for it in items if it['type'] == 'sentence']
    if sentence_items:
        tok_counts = [it['tokens'] for it in sentence_items]
        print(f"   📊 Token distribution per chunk:")
        print(f"      Min: {min(tok_counts)}  |  Max: {max(tok_counts)}"
              f"  |  Avg: {sum(tok_counts)//len(tok_counts)}")
        print(f"      Over-limit sentences fixed: {oversized}")
        print(f"      All chunks within {TOKEN_LIMIT} token limit: ✅")

    return items


# ══════════════════════════════════════════════════════════════
#  STEP 4 — Batch translate (never truncates)
# ══════════════════════════════════════════════════════════════

def translate_batch(sentences: list) -> list:
    """
    Translate a batch of sentences.
    truncation=False because ALL sentences are pre-verified to fit.
    """
    tokenizer.src_lang = SRC_LANG
    inputs = tokenizer(
        sentences,
        return_tensors="pt",
        padding=True,
        truncation=False,   # ← NO truncation! We pre-split safely
        max_length=1024,
    ).to(DEVICE)

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            forced_bos_token_id=_tgt_id,
            max_length=512,
            num_beams=NUM_BEAMS,
            early_stopping=True,
        )

    return tokenizer.batch_decode(generated, skip_special_tokens=True)


# ══════════════════════════════════════════════════════════════
#  STEP 5 — Full pipeline
# ══════════════════════════════════════════════════════════════

def translate_document(text: str) -> dict:
    print("=" * 65)
    print("  📄  Parsing document...")
    print("=" * 65)

    items = parse_document(text)

    sentence_items = [it for it in items if it['type'] == 'sentence']
    n_sents = len(sentence_items)
    n_paras = sum(1 for it in items if it['type'] == 'para_break') + 1
    sentences = [it['text'] for it in sentence_items]

    print(f"\n   Words      : {len(text.split())}")
    print(f"   Sentences  : {n_sents}")
    print(f"   Paragraphs : {n_paras}")
    print(f"   GPU Batches: {(n_sents + BATCH_SIZE - 1)//BATCH_SIZE}\n")

    # ── Translate in batches ──────────────────────────────────
    print("=" * 65)
    print("  ⚙️   Translating...")
    print("=" * 65)

    all_translations = []
    t_start = time.time()
    total_batches = (n_sents + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(0, n_sents, BATCH_SIZE):
        batch    = sentences[i : i + BATCH_SIZE]
        b_num    = i // BATCH_SIZE + 1
        t_b      = time.time()
        print(f"   Batch {b_num}/{total_batches}  {len(batch)} sentences...", end=" ")
        translated = translate_batch(batch)
        all_translations.extend(translated)
        print(f"✅ {time.time()-t_b:.2f}s")

    t_total = time.time() - t_start

    # ── Reconstruct document ──────────────────────────────────
    trans_iter = iter(all_translations)
    output_parts = []
    current_para = []

    for it in items:
        if it['type'] == 'para_break':
            if current_para:
                output_parts.append(' '.join(current_para))
                current_para = []
            output_parts.append('\n\n')
        else:
            current_para.append(next(trans_iter))

    if current_para:
        output_parts.append(' '.join(current_para))

    full_translation = ''.join(output_parts).strip()

    return {
        'translation'  : full_translation,
        'sentences_en' : sentences,
        'sentences_mr' : all_translations,
        'stats': {
            'words'   : len(text.split()),
            'sents'   : n_sents,
            'paras'   : n_paras,
            'time_s'  : round(t_total, 2),
            'wps'     : round(len(text.split()) / t_total, 1),
        }
    }


# ══════════════════════════════════════════════════════════════
#  PASTE YOUR 2-3 PAGES HERE
# ══════════════════════════════════════════════════════════════
document_text = """
The Future of Artificial Intelligence and Humanity

Artificial Intelligence (AI) has evolved from a speculative concept in science fiction to a transformative force shaping nearly every aspect of modern life. From recommendation systems and virtual assistants to autonomous vehicles and advanced medical diagnostics, AI is no longer a distant dream—it is a present reality. As we stand at the intersection of rapid technological advancement and societal change, it becomes increasingly important to explore how AI will influence the future of humanity.

At its core, artificial intelligence refers to the ability of machines to perform tasks that typically require human intelligence. These tasks include reasoning, learning, problem-solving, perception, and language understanding. Over the past decade, breakthroughs in machine learning, especially deep learning, have enabled systems to achieve remarkable performance in areas such as image recognition, natural language processing, and strategic gameplay. However, these advancements also raise important questions about ethics, employment, privacy, and the nature of intelligence itself.

One of the most significant impacts of AI is its potential to revolutionize industries. In healthcare, AI-powered systems can analyze vast amounts of medical data to assist in early diagnosis and personalized treatment plans. For example, algorithms can detect patterns in medical imaging that may be invisible to the human eye, enabling earlier detection of diseases such as cancer. Additionally, AI-driven drug discovery platforms are accelerating the development of new medications by predicting how different compounds will interact with biological systems. This could drastically reduce the time and cost associated with bringing new treatments to market.

In the realm of education, AI has the potential to create highly personalized learning experiences. Adaptive learning platforms can analyze a student’s strengths, weaknesses, and learning pace to tailor content accordingly. This could help bridge educational gaps and provide opportunities for learners in remote or underserved areas. Furthermore, AI tutors and virtual classrooms can offer 24/7 support, making education more accessible than ever before. However, the integration of AI in education also requires careful consideration of data privacy and the role of human educators in shaping critical thinking and social development.

The impact of AI on the workforce is another area of intense debate. Automation has already begun to replace repetitive and routine tasks in industries such as manufacturing, logistics, and customer service. While this can lead to increased efficiency and cost savings, it also raises concerns about job displacement. Many fear that AI will render certain professions obsolete, leading to widespread unemployment. However, history suggests that technological advancements often create new opportunities even as they eliminate old ones. The challenge lies in ensuring a smooth transition by investing in education, reskilling, and policies that support affected workers.

Moreover, AI is enabling entirely new categories of jobs and industries. Roles such as data scientists, machine learning engineers, and AI ethicists have emerged as critical components of the modern workforce. In addition, creative fields are being augmented by AI tools that assist in writing, music composition, design, and filmmaking. Rather than replacing human creativity, these tools can enhance it, allowing individuals to focus on higher-level thinking and innovation. The key is to view AI as a collaborator rather than a competitor.

Ethical considerations play a crucial role in shaping the future of AI. As systems become more autonomous and influential, questions about accountability, fairness, and transparency become increasingly important. Bias in AI algorithms is a well-documented issue, often arising from biased training data. This can lead to unfair outcomes in areas such as hiring, lending, and law enforcement. Addressing these challenges requires a multidisciplinary approach involving technologists, policymakers, and ethicists.

Another pressing concern is the issue of privacy. AI systems rely heavily on data, often personal and sensitive, to function effectively. The widespread collection and analysis of data raise questions about consent, surveillance, and data ownership. Striking a balance between innovation and privacy protection is essential to maintaining public trust. Regulations such as data protection laws and ethical guidelines can help ensure that AI is developed and deployed responsibly.

The concept of artificial general intelligence (AGI) adds another layer of complexity to the discussion. Unlike narrow AI, which is designed for specific tasks, AGI would possess the ability to understand, learn, and apply knowledge across a wide range of domains, similar to human intelligence. While AGI remains largely theoretical, its potential implications are profound. Some experts believe that achieving AGI could lead to unprecedented advancements in science, technology, and society. Others warn of existential risks if such systems are not aligned with human values.

The alignment problem—ensuring that AI systems act in accordance with human intentions and ethical principles—is one of the most critical challenges in AI research. As systems become more powerful, even small misalignments could have significant consequences. Researchers are exploring various approaches to address this issue, including reinforcement learning from human feedback, interpretability techniques, and robust safety mechanisms. The goal is to create AI systems that are not only intelligent but also trustworthy and aligned with societal goals.

AI’s influence extends beyond practical applications to philosophical questions about consciousness and identity. As machines become more capable of mimicking human behavior, it becomes increasingly difficult to distinguish between human and artificial intelligence. This raises questions about what it means to be human and whether machines could ever possess consciousness or self-awareness. While current AI systems do not have consciousness, ongoing advancements in neuroscience and cognitive science may provide insights into the nature of intelligence and the possibility of artificial consciousness.

The integration of AI into daily life also has cultural and social implications. AI-driven content recommendation systems shape how people consume information, influencing opinions, beliefs, and even social dynamics. While these systems can enhance user experience, they can also contribute to echo chambers and misinformation. Ensuring that AI promotes diverse perspectives and accurate information is essential for a healthy and informed society.

In addition, AI has the potential to address some of the world’s most pressing challenges. Climate change, for instance, can benefit from AI-driven models that predict environmental changes, optimize energy usage, and support sustainable practices. Similarly, AI can assist in disaster response by analyzing real-time data to coordinate relief efforts and minimize damage. These applications highlight the positive potential of AI when aligned with global priorities.

However, the global nature of AI development also raises questions about governance and international cooperation. Different countries have varying approaches to AI regulation, ethics, and deployment. Ensuring that AI is used responsibly on a global scale requires collaboration and shared standards. International organizations, governments, and private entities must work together to establish frameworks that promote innovation while safeguarding human rights.

Another important aspect of AI’s future is its accessibility. As AI technologies become more advanced, there is a risk that they may be concentrated in the hands of a few powerful organizations or nations. This could exacerbate existing inequalities and create new forms of digital divide. Democratizing access to AI tools and education is crucial to ensuring that the benefits of AI are distributed equitably. Open-source initiatives, public research funding, and inclusive policies can play a significant role in achieving this goal.

The role of human values in shaping AI cannot be overstated. Technology is not inherently good or bad; its impact depends on how it is designed and used. Embedding ethical considerations into the development process is essential to creating AI systems that serve humanity. This includes promoting fairness, accountability, transparency, and inclusivity. By prioritizing these values, society can harness the power of AI while minimizing potential risks.

Looking ahead, the relationship between humans and AI is likely to evolve in complex and unpredictable ways. Rather than a simple narrative of replacement or dominance, the future will likely involve a dynamic interplay between human and artificial intelligence. Humans bring creativity, empathy, and moral judgment, while AI offers speed, scalability, and analytical power. Together, they have the potential to achieve outcomes that neither could accomplish alone.

Education and awareness will play a critical role in preparing society for this future. Understanding how AI works, its limitations, and its implications is essential for informed decision-making. This includes not only technical knowledge but also ethical and societal perspectives. By fostering a culture of curiosity and critical thinking, individuals can better navigate the opportunities and challenges presented by AI.

In conclusion, the future of artificial intelligence and humanity is both promising and complex. AI has the potential to transform industries, improve quality of life, and address global challenges. At the same time, it raises important questions about ethics, employment, privacy, and the nature of intelligence. Navigating this landscape requires thoughtful consideration, collaboration, and a commitment to human values. By approaching AI with both optimism and responsibility, society can shape a future where technology serves as a force for good, enhancing human potential and creating a more equitable and sustainable world.
"""

# ── RUN ───────────────────────────────────────────────────────
result = translate_document(document_text)
# ══════════════════════════════════════════════════════════════
#  ACCURACY EVALUATION  — add after result = translate_document()
#  !pip install -q sacrebleu sentence-transformers deep-translator
# ══════════════════════════════════════════════════════════════
import sacrebleu, numpy as np, time
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Reuse sentences already extracted by translate_document() ─
en_sents = result['sentences_en']   # original English sentences
mr_nllb  = result['sentences_mr']   # NLLB Marathi translations

print("\n" + "="*65)
print("  🔬  ACCURACY EVALUATION")
print("="*65)
print(f"  Evaluating on {len(en_sents)} sentences from the document\n")

# ── Step 1: Google translate same sentences (for comparison) ──
print("🔴 [1/4] Google Translate (EN → MR)...")
gt_en_mr = GoogleTranslator(source="en", target="mr")
mr_google = []
for s in en_sents:
    try:    mr_google.append(gt_en_mr.translate(s) or "[empty]")
    except: mr_google.append("[error]")
    time.sleep(0.15)
print(f"   ✅ Done ({len(mr_google)} sentences)")

# ── Step 2: Back-translate both outputs to English ────────────
print("\n🔵 [2/4] Back-translating NLLB output  (MR → EN)...")
gt_mr_en = GoogleTranslator(source="mr", target="en")
bt_nllb = []
for s in mr_nllb:
    try:    bt_nllb.append(gt_mr_en.translate(s) or "[empty]")
    except: bt_nllb.append("[error]")
    time.sleep(0.15)

print("🔴 [3/4] Back-translating Google output (MR → EN)...")
bt_google = []
for s in mr_google:
    try:    bt_google.append(gt_mr_en.translate(s) or "[empty]")
    except: bt_google.append("[error]")
    time.sleep(0.15)
print("   ✅ Done")

# ── Step 3: Semantic similarity (EN original vs MR translation)
print("\n🧠 [4/4] Computing semantic similarity...")
emb_model  = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
en_embs    = emb_model.encode(en_sents)
nllb_embs  = emb_model.encode(mr_nllb)
goog_embs  = emb_model.encode(mr_google)

nllb_sims  = [float(cosine_similarity([e],[n])[0][0])
              for e,n in zip(en_embs, nllb_embs)]
goog_sims  = [float(cosine_similarity([e],[g])[0][0])
              for e,g in zip(en_embs, goog_embs)]
print("   ✅ Done")

# ── Step 4: Compute metrics ───────────────────────────────────
nllb_bleu  = sacrebleu.corpus_bleu(bt_nllb,   [en_sents]).score
goog_bleu  = sacrebleu.corpus_bleu(bt_google, [en_sents]).score
nllb_chrf  = sacrebleu.corpus_chrf(mr_nllb,   [mr_google]).score
nllb_avg   = float(np.mean(nllb_sims))
goog_avg   = float(np.mean(goog_sims))
nllb_wins  = sum(n > g for n,g in zip(nllb_sims, goog_sims))
goog_wins  = sum(g > n for n,g in zip(nllb_sims, goog_sims))

# ── Results ───────────────────────────────────────────────────
print("\n" + "═"*65)
print("  🏆  ACCURACY SCORECARD")
print("═"*65)
print(f"\n  {'Metric':<38} {'🔵 NLLB':>8}  {'🔴 Google':>9}  Winner")
print("  " + "─"*60)

rows = [
    ("Back-Translation BLEU  (↑)",  nllb_bleu, goog_bleu, "higher"),
    ("Avg Semantic Similarity (↑)", nllb_avg,  goog_avg,  "higher"),
    ("Sentence Wins          (↑)",  nllb_wins, goog_wins, "higher"),
]
nllb_score = goo_score = 0
for name, nv, gv, _ in rows:
    w = "🔵 NLLB" if nv >= gv else "🔴 Google"
    if nv >= gv: nllb_score += 1
    else:        goo_score  += 1
    print(f"  {name:<38} {nv:>8.3f}  {gv:>9.3f}  {w}")

print("  " + "─"*60)
print(f"  {'ChrF NLLB vs Google (similarity)':<38} {nllb_chrf:>8.2f}"
      f"  {'(80+=very similar)'}")
print(f"\n  Overall  →  🔵 NLLB [{nllb_score}/3]  vs  🔴 Google [{goo_score}/3]")
overall = "🔵 NLLB-200" if nllb_score >= goo_score else "🔴 Google Translate"
print(f"  🏆 Winner : {overall}")
print("═"*65)

# ── Sample comparisons (first 5 sentences) ───────────────────
print("\n  📋  SAMPLE SENTENCES (first 5)\n")
for i, (en, n_mr, g_mr, n_bt, g_bt, n_s, g_s) in enumerate(zip(
    en_sents[:5], mr_nllb[:5], mr_google[:5],
    bt_nllb[:5],  bt_google[:5],
    nllb_sims[:5], goog_sims[:5]
), 1):
    win = "🔵" if n_s > g_s else ("🔴" if g_s > n_s else "🟡")
    print(f"  [{i:02d}] {win}  {en}")
    print(f"        🔵 NLLB   : {n_mr}")
    print(f"             back : {n_bt}  [sim={n_s:.3f}]")
    print(f"        🔴 Google : {g_mr}")
    print(f"             back : {g_bt}  [sim={g_s:.3f}]")
    print()

print("  ℹ️  Back-trans BLEU: how well meaning is preserved after MR→EN round-trip")
print("  ℹ️  Semantic Sim  : cosine similarity of multilingual embeddings (EN vs MR)")
print("  ℹ️  ChrF          : character overlap between NLLB & Google (80+=near identical)\n")
# print("\n" + "═" * 65)
# print("  🇮🇳  MARATHI TRANSLATION")
# print("═" * 65 + "\n")
# print(result['translation'])

# s = result['stats']
# print("\n" + "═" * 65)
# print(f"  ✅  Done!  {s['words']} words in {s['time_s']}s  ({s['wps']} words/sec)")
# print("═" * 65)

🖥️  Device : CUDA
⏳ Loading model...


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

✅ Model ready in 14.6s

  📄  Parsing document...
   📊 Token distribution per chunk:
      Min: 10  |  Max: 44  |  Avg: 26
      Over-limit sentences fixed: 0
      All chunks within 400 token limit: ✅

   Words      : 1459
   Sentences  : 87
   Paragraphs : 20
   GPU Batches: 6

  ⚙️   Translating...
   Batch 1/6  16 sentences... ✅ 21.75s
   Batch 2/6  16 sentences... ✅ 1.58s
   Batch 3/6  16 sentences... ✅ 1.43s
   Batch 4/6  16 sentences... ✅ 1.55s
   Batch 5/6  16 sentences... ✅ 1.15s
   Batch 6/6  7 sentences... ✅ 1.34s

  🔬  ACCURACY EVALUATION
  Evaluating on 87 sentences from the document

🔴 [1/4] Google Translate (EN → MR)...
   ✅ Done (87 sentences)

🔵 [2/4] Back-translating NLLB output  (MR → EN)...
🔴 [3/4] Back-translating Google output (MR → EN)...
   ✅ Done

🧠 [4/4] Computing semantic similarity...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   ✅ Done

═════════════════════════════════════════════════════════════════
  🏆  ACCURACY SCORECARD
═════════════════════════════════════════════════════════════════

  Metric                                   🔵 NLLB   🔴 Google  Winner
  ────────────────────────────────────────────────────────────
  Back-Translation BLEU  (↑)               74.273     72.679  🔵 NLLB
  Avg Semantic Similarity (↑)               0.983      0.888  🔵 NLLB
  Sentence Wins          (↑)               85.000      2.000  🔵 NLLB
  ────────────────────────────────────────────────────────────
  ChrF NLLB vs Google (similarity)           0.56  (80+=very similar)

  Overall  →  🔵 NLLB [3/3]  vs  🔴 Google [0/3]
  🏆 Winner : 🔵 NLLB-200
═════════════════════════════════════════════════════════════════

  📋  SAMPLE SENTENCES (first 5)

  [01] 🔴  The Future of Artificial Intelligence and Humanity
        🔵 NLLB   : Artificial Artificial Artificial Artificial Artificial Artificial Artificial Artificial Artificial Artificia

 request for WORDS PER

In [ ]:
# ══════════════════════════════════════════════════════════════
#  DUAL-MODEL BULK TRANSLATOR
#  Runs NLLB-600M  AND  mBART-large-50 back-to-back,
#  then compares both vs Google Translate
#  Target: 2000 WPM in production
# ══════════════════════════════════════════════════════════════

import gc
import re
import time
import warnings
import torch
import numpy as np
import sacrebleu
warnings.filterwarnings("ignore")

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

try:
    import nltk
    nltk.download('punkt_tab', quiet=True)
    from nltk.tokenize import sent_tokenize
    HAS_NLTK = True
except Exception:
    HAS_NLTK = False

# ══════════════════════════════════════════════════════════════
#  GLOBAL CONFIG
# ══════════════════════════════════════════════════════════════

DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 16
NUM_BEAMS   = 4
TOKEN_LIMIT = 400   # safe per-chunk limit

TARGET_WPM  = 2000  # production target

# Both models to run + their correct language codes
MODELS = [
    {
        "label"     : "NLLB-600M",
        "name"      : "facebook/nllb-200-distilled-600M",
        "src_lang"  : "eng_Latn",
        "tgt_lang"  : "mar_Deva",
        "emoji"     : "🟢",
    },
    {
        "label"     : "mBART-large-50",
        "name"      : "facebook/mbart-large-50-many-to-many-mmt",
        "src_lang"  : "en_XX",        # ← correct code for mBART
        "tgt_lang"  : "mr_IN",        # ← correct code for mBART
        "emoji"     : "🔵",
    },
]

# ══════════════════════════════════════════════════════════════
#  GPU HELPERS
# ══════════════════════════════════════════════════════════════

GPU_REFERENCE_SPEEDS = {
    "RTX 3060"    :  320,  "RTX 3080"    :  580,
    "RTX 3090"    :  820,  "RTX 4080"    : 1050,
    "RTX 4090"    : 1500,  "A10G"        : 1100,
    "A100 (40GB)" : 2200,  "A100 (80GB)" : 2400,
    "H100"        : 4500,  "T4"          :  380,
    "V100 (16GB)" :  700,  "V100 (32GB)" :  750,
    "L4"          :  900,  "CPU (no GPU)":   35,
}


def get_gpu_info() -> dict:
    info = {
        "device": DEVICE.upper(), "gpu_name": "N/A (CPU only)",
        "vram_gb": 0.0, "cuda_version": "N/A",
        "gpu_count": 0, "compute_cap": "N/A",
    }
    if not torch.cuda.is_available():
        return info
    info["gpu_count"]   = torch.cuda.device_count()
    info["gpu_name"]    = torch.cuda.get_device_name(0)
    info["vram_gb"]     = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    info["cuda_version"]= torch.version.cuda or "N/A"
    cap = torch.cuda.get_device_capability(0)
    info["compute_cap"] = f"{cap[0]}.{cap[1]}"
    return info


def print_system_banner(gpu: dict):
    print("\n" + "═"*65)
    print("  🖥️   SYSTEM CONFIGURATION")
    print("═"*65)
    print(f"  Device       : {gpu['device']}")
    print(f"  GPU          : {gpu['gpu_name']}")
    print(f"  VRAM         : {gpu['vram_gb']} GB")
    print(f"  CUDA         : {gpu['cuda_version']}")
    print(f"  GPU Count    : {gpu['gpu_count']}")
    print(f"  Compute Cap. : {gpu['compute_cap']}")
    print(f"  Batch Size   : {BATCH_SIZE}  |  Beams: {NUM_BEAMS}")
    print(f"  Models       : {' vs '.join(m['label'] for m in MODELS)}")
    print("═"*65 + "\n")


# ══════════════════════════════════════════════════════════════
#  DOCUMENT PARSING  (once, shared by both models)
# ══════════════════════════════════════════════════════════════

SPLIT_PATTERNS = [
    r'(?<=\.\s)', r'(?<=;\s)', r'(?<=,\s)',
    r'\s+(?=and|but|or|however|therefore|moreover|furthermore|whereas)\s+',
    r'\s+',
]


def _count_tokens_raw(tokenizer, src_lang: str, text: str) -> int:
    tokenizer.src_lang = src_lang
    return len(tokenizer(text, add_special_tokens=True)["input_ids"])


def _sub_split(tokenizer, src_lang, sentence, token_limit):
    if _count_tokens_raw(tokenizer, src_lang, sentence) <= token_limit:
        return [sentence]
    for pattern in SPLIT_PATTERNS:
        parts = [p.strip() for p in re.split(pattern, sentence) if p.strip()]
        if len(parts) <= 1:
            continue
        safe = []
        for p in parts:
            if _count_tokens_raw(tokenizer, src_lang, p) <= token_limit:
                safe.append(p)
            else:
                safe.extend(_sub_split(tokenizer, src_lang, p, token_limit))
        if safe:
            return safe
    words = sentence.split()
    mid = len(words) // 2
    return (_sub_split(tokenizer, src_lang, ' '.join(words[:mid]), token_limit) +
            _sub_split(tokenizer, src_lang, ' '.join(words[mid:]), token_limit))


def parse_document(text: str, tokenizer, src_lang: str) -> list:
    """
    Parse text into flat list of {'type':'sentence','text':...} /
    {'type':'para_break'} items.  Every sentence fits within TOKEN_LIMIT.
    """
    items, oversized = [], 0
    paragraphs = re.split(r'\n\s*\n', text.strip())

    for p_idx, para in enumerate(paragraphs):
        if p_idx > 0:
            items.append({'type': 'para_break'})
        para = para.strip()
        if not para:
            continue

        raw_sents = (sent_tokenize(para, language='english')
                     if HAS_NLTK else re.split(r'(?<=[.!?])\s+', para))

        for raw in raw_sents:
            raw = raw.strip()
            if not raw:
                continue
            n = _count_tokens_raw(tokenizer, src_lang, raw)
            if n <= TOKEN_LIMIT:
                items.append({'type': 'sentence', 'text': raw, 'tokens': n})
            else:
                oversized += 1
                for chunk in _sub_split(tokenizer, src_lang, raw, TOKEN_LIMIT):
                    items.append({
                        'type': 'sentence', 'text': chunk,
                        'tokens': _count_tokens_raw(tokenizer, src_lang, chunk)
                    })

    sent_items = [it for it in items if it['type'] == 'sentence']
    if sent_items:
        tc = [it['tokens'] for it in sent_items]
        print(f"   📊 Chunks: {len(sent_items)}  |  "
              f"Min: {min(tc)}  Max: {max(tc)}  Avg: {sum(tc)//len(tc)} tokens")
        print(f"      Over-limit fixed: {oversized}  |  All ≤ {TOKEN_LIMIT}: ✅")
    return items


# ══════════════════════════════════════════════════════════════
#  SINGLE-MODEL TRANSLATION PIPELINE
# ══════════════════════════════════════════════════════════════

def translate_with_model(cfg: dict, text: str) -> dict:
    """
    Load model → parse → translate → unload from VRAM.
    Returns result dict including stats and per-sentence outputs.
    """
    label    = cfg["label"]
    mname    = cfg["name"]
    src_lang = cfg["src_lang"]
    tgt_lang = cfg["tgt_lang"]
    emoji    = cfg["emoji"]

    print("\n" + "═"*65)
    print(f"  {emoji}  RUNNING : {label}")
    print(f"      Model   : {mname}")
    print(f"      Lang    : {src_lang} → {tgt_lang}")
    print("═"*65)

    # ── Load ──────────────────────────────────────────────────
    print("⏳ Loading model...")
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(mname)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        mname,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
    ).to(DEVICE)
    model.eval()
    tokenizer.src_lang = src_lang
    tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    print(f"✅ Model ready in {time.time()-t0:.1f}s\n")

    # ── Parse ─────────────────────────────────────────────────
    print("=" * 65)
    print("  📄  Parsing document...")
    print("=" * 65)
    items       = parse_document(text, tokenizer, src_lang)
    sent_items  = [it for it in items if it['type'] == 'sentence']
    n_sents     = len(sent_items)
    n_paras     = sum(1 for it in items if it['type'] == 'para_break') + 1
    sentences   = [it['text'] for it in sent_items]
    n_words     = len(text.split())

    print(f"\n   Words      : {n_words:,}")
    print(f"   Sentences  : {n_sents}")
    print(f"   Paragraphs : {n_paras}")
    print(f"   GPU Batches: {(n_sents + BATCH_SIZE - 1)//BATCH_SIZE}\n")

    # ── Translate ─────────────────────────────────────────────
    print("=" * 65)
    print("  ⚙️   Translating...")
    print("=" * 65)

    def _batch(sents):
        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            sents, return_tensors="pt", padding=True,
            truncation=False, max_length=1024,
        ).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=tgt_id,
                max_length=512,
                num_beams=NUM_BEAMS,
                early_stopping=True,
            )
        return tokenizer.batch_decode(out, skip_special_tokens=True)

    all_translations = []
    t_start      = time.time()
    total_batches = (n_sents + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(0, n_sents, BATCH_SIZE):
        batch = sentences[i: i + BATCH_SIZE]
        b_num = i // BATCH_SIZE + 1
        t_b   = time.time()
        print(f"   Batch {b_num}/{total_batches}  ({len(batch)} sents)...", end=" ")
        all_translations.extend(_batch(batch))
        print(f"✅ {time.time()-t_b:.2f}s")

    t_total = time.time() - t_start

    # ── Reconstruct ───────────────────────────────────────────
    trans_iter   = iter(all_translations)
    output_parts = []
    current_para = []
    for it in items:
        if it['type'] == 'para_break':
            if current_para:
                output_parts.append(' '.join(current_para))
                current_para = []
            output_parts.append('\n\n')
        else:
            current_para.append(next(trans_iter))
    if current_para:
        output_parts.append(' '.join(current_para))

    # ── Unload VRAM ───────────────────────────────────────────
    print(f"\n🗑️  Unloading {label} from VRAM...")
    del model
    del tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    print("   ✅ VRAM freed\n")

    stats = {
        'words'  : n_words,
        'sents'  : n_sents,
        'paras'  : n_paras,
        'time_s' : round(t_total, 2),
        'wps'    : round(n_words / t_total, 1),
        'wpm'    : round(n_words / t_total * 60, 1),
    }

    return {
        'label'        : label,
        'emoji'        : emoji,
        'translation'  : ''.join(output_parts).strip(),
        'sentences_en' : sentences,
        'sentences_mr' : all_translations,
        'stats'        : stats,
    }


# ══════════════════════════════════════════════════════════════
#  SPEED REPORT (per model)
# ══════════════════════════════════════════════════════════════

def print_speed_report(result: dict, gpu: dict):
    stats  = result['stats']
    wpm    = stats['wpm']
    pct    = round(wpm / TARGET_WPM * 100, 1)
    filled = min(30, int(30 * wpm / TARGET_WPM))
    bar    = "█" * filled + "░" * (30 - filled)
    icon   = "✅" if pct >= 100 else ("⚠️ " if pct >= 50 else "🔴")
    proj   = max(1, -(-TARGET_WPM // int(wpm))) if wpm > 0 else "?"

    print(f"\n  ⚡ {result['emoji']} {result['label']} — Speed")
    print(f"  ─────────────────────────────────────────────────────")
    print(f"  Time Taken        : {stats['time_s']}s")
    print(f"  Words / Minute    : {wpm:,.1f} WPM")
    print(f"  Words / Second    : {stats['wps']:,.1f} WPS")
    print(f"  vs {TARGET_WPM} WPM target  : [{bar}] {pct}%  {icon}")
    if pct >= 100:
        print(f"  🎉 TARGET MET on single {gpu['gpu_name']}!")
    else:
        print(f"  Need ~{proj}× {gpu['gpu_name']} to reach {TARGET_WPM} WPM")


# ══════════════════════════════════════════════════════════════
#  ACCURACY EVALUATION  (3-way: NLLB vs mBART vs Google)
# ══════════════════════════════════════════════════════════════

def run_accuracy_evaluation(results: list, gpu: dict):
    """
    results = list of dicts from translate_with_model()
    Compares all models vs Google Translate + head-to-head.
    """
    en_sents = results[0]['sentences_en']   # same for both (same parse)
    n = len(en_sents)

    print("\n" + "="*65)
    print("  🔬  ACCURACY EVALUATION  (3-way comparison)")
    print("="*65)
    print(f"  Sentences: {n}  |  Models: {' vs '.join(r['label'] for r in results)}\n")

    # ── Google Translate (EN → MR) ────────────────────────────
    print("🔴 [1] Google Translate  EN → MR ...")
    gt_en = GoogleTranslator(source="en", target="mr")
    mr_google = []
    for s in en_sents:
        try:    mr_google.append(gt_en.translate(s) or "[empty]")
        except: mr_google.append("[error]")
        time.sleep(0.15)
    print(f"      ✅ Done ({len(mr_google)} sentences)\n")

    # ── Back-translate all MR outputs → EN ───────────────────
    gt_mr = GoogleTranslator(source="mr", target="en")

    def back_translate(mr_sents, tag):
        print(f"   Back-translating {tag} (MR → EN)...")
        bt = []
        for s in mr_sents:
            try:    bt.append(gt_mr.translate(s) or "[empty]")
            except: bt.append("[error]")
            time.sleep(0.15)
        return bt

    bt_google = back_translate(mr_google, "Google")

    for r in results:
        r['bt'] = back_translate(r['sentences_mr'], r['label'])

    print()

    # ── Semantic similarity ───────────────────────────────────
    print("🧠 Computing semantic similarity (multilingual embeddings)...")
    emb_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
    en_embs   = emb_model.encode(en_sents)
    g_embs    = emb_model.encode(mr_google)
    g_sims    = [float(cosine_similarity([e], [g])[0][0])
                 for e, g in zip(en_embs, g_embs)]

    for r in results:
        m_embs = emb_model.encode(r['sentences_mr'])
        r['sims'] = [float(cosine_similarity([e], [m])[0][0])
                     for e, m in zip(en_embs, m_embs)]
    print("      ✅ Done\n")

    # ── Metrics ───────────────────────────────────────────────
    g_bleu = sacrebleu.corpus_bleu(bt_google, [en_sents]).score
    g_avg  = float(np.mean(g_sims))

    for r in results:
        r['bleu']  = sacrebleu.corpus_bleu(r['bt'], [en_sents]).score
        r['chrf']  = sacrebleu.corpus_chrf(r['sentences_mr'], [mr_google]).score
        r['avg_sim'] = float(np.mean(r['sims']))
        r['wins_vs_google'] = sum(s > g for s, g in zip(r['sims'], g_sims))

    # ══════════════════════════════════════════════════════════
    #  SPEED COMPARISON TABLE
    # ══════════════════════════════════════════════════════════
    print("\n" + "═"*65)
    print("  ⚡  SPEED COMPARISON  (vs 2,000 WPM Production Target)")
    print("═"*65)
    print(f"\n  {'Model':<20} {'WPM':>9}  {'WPS':>6}  {'Time':>7}  {'vs Target':>12}")
    print("  " + "─"*58)
    for r in results:
        s   = r['stats']
        pct = round(s['wpm'] / TARGET_WPM * 100, 1)
        icon = "✅" if pct >= 100 else "❌"
        print(f"  {r['emoji']} {r['label']:<18} {s['wpm']:>9,.1f}"
              f"  {s['wps']:>6,.1f}  {s['time_s']:>6.2f}s"
              f"  {pct:>7.1f}%  {icon}")
    print(f"  🔴 {'Google Translate':<18} {'API':>9}  {'—':>6}  {'—':>7}  {'Ref':>12}")
    print("  " + "─"*58)

    fastest = max(results, key=lambda r: r['stats']['wpm'])
    print(f"\n  🏆 Fastest Model : {fastest['emoji']} {fastest['label']} "
          f"({fastest['stats']['wpm']:,.0f} WPM)")

    # ── GPU reference table (once) ────────────────────────────
    print(f"\n  ── Reference GPU Speeds (NLLB-600M, beam=4) ───────")
    print(f"  {'GPU':<28} {'Est. WPM':>9}  {'≥2K':>5}")
    print(f"  {'─'*28}  {'─'*9}  {'─'*5}")
    for gname, wpm in GPU_REFERENCE_SPEEDS.items():
        meets  = "✅" if wpm >= TARGET_WPM else "  "
        marker = " ◄ YOU" if gname in gpu["gpu_name"] else ""
        print(f"  {gname:<28} {wpm:>9,}  {meets}{marker}")

    # ══════════════════════════════════════════════════════════
    #  ACCURACY COMPARISON TABLE
    # ══════════════════════════════════════════════════════════
    print("\n" + "═"*65)
    print("  🏆  ACCURACY SCORECARD  (vs Google Translate)")
    print("═"*65)

    headers = ["Metric", "🔴 Google"] + [f"{r['emoji']} {r['label']}" for r in results]
    col_w   = 14
    hdr_row = f"  {'Metric':<30}" + "".join(f"{h:>{col_w}}" for h in headers[1:])
    print(f"\n{hdr_row}")
    print("  " + "─"*70)

    def row(name, g_val, r_vals, fmt=".3f"):
        vals = [g_val] + r_vals
        best = max(vals)
        cells = []
        for v in vals:
            marker = " ★" if v == best else "  "
            cells.append(f"{v:{col_w-2}{fmt}}{marker}")
        return f"  {name:<30}" + "".join(cells)

    print(row("Back-Trans BLEU  (↑)",
              g_bleu,  [r['bleu']     for r in results]))
    print(row("Avg Semantic Sim (↑)",
              g_avg,   [r['avg_sim']  for r in results]))
    print(row("ChrF vs Google   (↑)",
              100.0,   [r['chrf']     for r in results]))

    print("\n  Sentence-level wins vs Google:")
    for r in results:
        print(f"  {r['emoji']} {r['label']:<18} wins {r['wins_vs_google']}/{n} sentences")

    # Head-to-head between the two models
    if len(results) == 2:
        r1, r2 = results
        h2h_r1 = sum(a > b for a, b in zip(r1['sims'], r2['sims']))
        h2h_r2 = n - h2h_r1
        print(f"\n  Head-to-head (semantic sim):")
        print(f"  {r1['emoji']} {r1['label']} wins {h2h_r1}/{n}  vs  "
              f"{r2['emoji']} {r2['label']} wins {h2h_r2}/{n}")

    # Overall quality winner
    def score(r):
        return (r['bleu'] + r['avg_sim'] * 100 + r['chrf']) / 3

    quality_winner = max(results, key=score)
    speed_winner   = fastest
    print(f"\n  🏆 Quality Winner : {quality_winner['emoji']} {quality_winner['label']}")
    print(f"  ⚡ Speed Winner   : {speed_winner['emoji']} {speed_winner['label']}")

    # Recommendation
    print("\n" + "═"*65)
    print("  📌  PRODUCTION RECOMMENDATION")
    print("═"*65)
    print(f"\n  Both models run on a single Tesla T4 (15.6 GB VRAM).")
    for r in results:
        s   = r['stats']
        pct = round(s['wpm'] / TARGET_WPM * 100, 1)
        q   = round(score(r), 2)
        print(f"\n  {r['emoji']} {r['label']}")
        print(f"     Speed   : {s['wpm']:,.0f} WPM  ({pct}% of target)")
        print(f"     Quality : composite score {q:.2f}")
        print(f"     ChrF vs Google : {r['chrf']:.1f}  (80+ = near-identical)")
    print()

    # ── Sample comparisons ────────────────────────────────────
    print("═"*65)
    print("  📋  SAMPLE SENTENCES (first 5)")
    print("═"*65 + "\n")

    for i, en in enumerate(en_sents[:5], 1):
        sims   = [r['sims'][i-1] for r in results]
        best_i = sims.index(max(sims))
        print(f"  [{i:02d}] {en}")
        for r in results:
            mr  = r['sentences_mr'][i-1]
            bt  = r['bt'][i-1]
            sim = r['sims'][i-1]
            tag = " ★ BEST" if r == results[best_i] else ""
            print(f"        {r['emoji']} {r['label']:<16}: {mr}")
            print(f"             back : {bt}  [sim={sim:.3f}]{tag}")
        g_mr = mr_google[i-1]
        g_bt = bt_google[i-1]
        g_s  = g_sims[i-1]
        print(f"        🔴 Google          : {g_mr}")
        print(f"             back : {g_bt}  [sim={g_s:.3f}]")
        print()

    print("  ℹ️  Back-trans BLEU : meaning preservation after MR→EN round-trip")
    print("  ℹ️  Semantic Sim    : multilingual embedding cosine similarity")
    print("  ℹ️  ChrF            : character n-gram overlap vs Google Translate")
    print("  ℹ️  ★               : best score for that sentence/metric\n")


# ══════════════════════════════════════════════════════════════
#  DOCUMENT TEXT — paste your content here
# ══════════════════════════════════════════════════════════════

document_text = """
The Future of Artificial Intelligence and Humanity

Artificial Intelligence (AI) has evolved from a speculative concept in science fiction to a transformative force shaping nearly every aspect of modern life. From recommendation systems and virtual assistants to autonomous vehicles and advanced medical diagnostics, AI is no longer a distant dream—it is a present reality. As we stand at the intersection of rapid technological advancement and societal change, it becomes increasingly important to explore how AI will influence the future of humanity.

At its core, artificial intelligence refers to the ability of machines to perform tasks that typically require human intelligence. These tasks include reasoning, learning, problem-solving, perception, and language understanding. Over the past decade, breakthroughs in machine learning, especially deep learning, have enabled systems to achieve remarkable performance in areas such as image recognition, natural language processing, and strategic gameplay. However, these advancements also raise important questions about ethics, employment, privacy, and the nature of intelligence itself.

One of the most significant impacts of AI is its potential to revolutionize industries. In healthcare, AI-powered systems can analyze vast amounts of medical data to assist in early diagnosis and personalized treatment plans. For example, algorithms can detect patterns in medical imaging that may be invisible to the human eye, enabling earlier detection of diseases such as cancer. Additionally, AI-driven drug discovery platforms are accelerating the development of new medications by predicting how different compounds will interact with biological systems. This could drastically reduce the time and cost associated with bringing new treatments to market.

In the realm of education, AI has the potential to create highly personalized learning experiences. Adaptive learning platforms can analyze a student's strengths, weaknesses, and learning pace to tailor content accordingly. This could help bridge educational gaps and provide opportunities for learners in remote or underserved areas. Furthermore, AI tutors and virtual classrooms can offer 24/7 support, making education more accessible than ever before. However, the integration of AI in education also requires careful consideration of data privacy and the role of human educators in shaping critical thinking and social development.

The impact of AI on the workforce is another area of intense debate. Automation has already begun to replace repetitive and routine tasks in industries such as manufacturing, logistics, and customer service. While this can lead to increased efficiency and cost savings, it also raises concerns about job displacement. Many fear that AI will render certain professions obsolete, leading to widespread unemployment. However, history suggests that technological advancements often create new opportunities even as they eliminate old ones. The challenge lies in ensuring a smooth transition by investing in education, reskilling, and policies that support affected workers.

Moreover, AI is enabling entirely new categories of jobs and industries. Roles such as data scientists, machine learning engineers, and AI ethicists have emerged as critical components of the modern workforce. In addition, creative fields are being augmented by AI tools that assist in writing, music composition, design, and filmmaking. Rather than replacing human creativity, these tools can enhance it, allowing individuals to focus on higher-level thinking and innovation. The key is to view AI as a collaborator rather than a competitor.

Ethical considerations play a crucial role in shaping the future of AI. As systems become more autonomous and influential, questions about accountability, fairness, and transparency become increasingly important. Bias in AI algorithms is a well-documented issue, often arising from biased training data. This can lead to unfair outcomes in areas such as hiring, lending, and law enforcement. Addressing these challenges requires a multidisciplinary approach involving technologists, policymakers, and ethicists.

Another pressing concern is the issue of privacy. AI systems rely heavily on data, often personal and sensitive, to function effectively. The widespread collection and analysis of data raise questions about consent, surveillance, and data ownership. Striking a balance between innovation and privacy protection is essential to maintaining public trust. Regulations such as data protection laws and ethical guidelines can help ensure that AI is developed and deployed responsibly.

The concept of artificial general intelligence (AGI) adds another layer of complexity to the discussion. Unlike narrow AI, which is designed for specific tasks, AGI would possess the ability to understand, learn, and apply knowledge across a wide range of domains, similar to human intelligence. While AGI remains largely theoretical, its potential implications are profound. Some experts believe that achieving AGI could lead to unprecedented advancements in science, technology, and society. Others warn of existential risks if such systems are not aligned with human values.

The alignment problem—ensuring that AI systems act in accordance with human intentions and ethical principles—is one of the most critical challenges in AI research. As systems become more powerful, even small misalignments could have significant consequences. Researchers are exploring various approaches to address this issue, including reinforcement learning from human feedback, interpretability techniques, and robust safety mechanisms. The goal is to create AI systems that are not only intelligent but also trustworthy and aligned with societal goals.

AI's influence extends beyond practical applications to philosophical questions about consciousness and identity. As machines become more capable of mimicking human behavior, it becomes increasingly difficult to distinguish between human and artificial intelligence. This raises questions about what it means to be human and whether machines could ever possess consciousness or self-awareness. While current AI systems do not have consciousness, ongoing advancements in neuroscience and cognitive science may provide insights into the nature of intelligence and the possibility of artificial consciousness.

The integration of AI into daily life also has cultural and social implications. AI-driven content recommendation systems shape how people consume information, influencing opinions, beliefs, and even social dynamics. While these systems can enhance user experience, they can also contribute to echo chambers and misinformation. Ensuring that AI promotes diverse perspectives and accurate information is essential for a healthy and informed society.

In addition, AI has the potential to address some of the world's most pressing challenges. Climate change, for instance, can benefit from AI-driven models that predict environmental changes, optimize energy usage, and support sustainable practices. Similarly, AI can assist in disaster response by analyzing real-time data to coordinate relief efforts and minimize damage. These applications highlight the positive potential of AI when aligned with global priorities.

However, the global nature of AI development also raises questions about governance and international cooperation. Different countries have varying approaches to AI regulation, ethics, and deployment. Ensuring that AI is used responsibly on a global scale requires collaboration and shared standards. International organizations, governments, and private entities must work together to establish frameworks that promote innovation while safeguarding human rights.

Another important aspect of AI's future is its accessibility. As AI technologies become more advanced, there is a risk that they may be concentrated in the hands of a few powerful organizations or nations. This could exacerbate existing inequalities and create new forms of digital divide. Democratizing access to AI tools and education is crucial to ensuring that the benefits of AI are distributed equitably. Open-source initiatives, public research funding, and inclusive policies can play a significant role in achieving this goal.

The role of human values in shaping AI cannot be overstated. Technology is not inherently good or bad; its impact depends on how it is designed and used. Embedding ethical considerations into the development process is essential to creating AI systems that serve humanity. This includes promoting fairness, accountability, transparency, and inclusivity. By prioritizing these values, society can harness the power of AI while minimizing potential risks.

Looking ahead, the relationship between humans and AI is likely to evolve in complex and unpredictable ways. Rather than a simple narrative of replacement or dominance, the future will likely involve a dynamic interplay between human and artificial intelligence. Humans bring creativity, empathy, and moral judgment, while AI offers speed, scalability, and analytical power. Together, they have the potential to achieve outcomes that neither could accomplish alone.

Education and awareness will play a critical role in preparing society for this future. Understanding how AI works, its limitations, and its implications is essential for informed decision-making. This includes not only technical knowledge but also ethical and societal perspectives. By fostering a culture of curiosity and critical thinking, individuals can better navigate the opportunities and challenges presented by AI.

In conclusion, the future of artificial intelligence and humanity is both promising and complex. AI has the potential to transform industries, improve quality of life, and address global challenges. At the same time, it raises important questions about ethics, employment, privacy, and the nature of intelligence. Navigating this landscape requires thoughtful consideration, collaboration, and a commitment to human values. By approaching AI with both optimism and responsibility, society can shape a future where technology serves as a force for good, enhancing human potential and creating a more equitable and sustainable world.



The idea of resilience has fascinated thinkers, philosophers, and ordinary people for centuries, not because it is rare, but because it is so deeply human and yet so unevenly distributed in how it appears across different lives and circumstances; resilience is often misunderstood as a kind of loud, visible strength, something that roars in the face of adversity, but more often it is quiet, almost invisible, a slow and steady persistence that refuses to give up even when there is no clear reason to continue, and it is in this quiet persistence that the true nature of resilience reveals itself, not as an extraordinary trait possessed by a few, but as a potential embedded within everyone, waiting to be recognized, nurtured, and expressed in moments of difficulty; when people imagine resilience, they often think of dramatic stories of survival, tales where individuals overcome overwhelming odds, but the more common and perhaps more meaningful expressions of resilience are found in everyday life, in the person who wakes up each day despite feeling defeated, in the student who continues to study despite repeated failures, in the worker who keeps striving in an environment that offers little encouragement, and in these small acts of continuation lies a profound strength that does not seek recognition but nonetheless shapes the course of lives in powerful ways; the development of resilience is not a simple or linear process, and it does not arise solely from hardship, as is often assumed, but from the interplay between challenge and support, between struggle and meaning, and between failure and reflection, and it is through this complex interaction that individuals begin to build an inner framework that allows them to endure and adapt, rather than simply react or collapse under pressure; one of the key elements of resilience is the ability to reframe experiences, to look at a situation not only for what it takes away but also for what it reveals or teaches, and while this does not diminish the pain or difficulty of the experience, it allows the individual to integrate it into a larger narrative of growth, rather than seeing it as a defining endpoint, and this shift in perspective can transform even deeply negative experiences into sources of insight and strength, though it often takes time, patience, and sometimes guidance to reach such a point; another important aspect of resilience is emotional regulation, the capacity to experience intense feelings without being overwhelmed by them, to acknowledge fear, sadness, anger, or frustration without allowing these emotions to dictate actions entirely, and this does not mean suppressing emotions, but rather developing a relationship with them that is neither avoidant nor overly reactive, and this balance enables individuals to respond thoughtfully rather than impulsively, which is a crucial factor in navigating difficult situations effectively; social connections also play a vital role in resilience, as humans are inherently relational beings, and the presence of even a small support system can make a significant difference in how challenges are perceived and managed, whether it is a friend who listens without judgment, a mentor who provides guidance, or a family member who offers unconditional support, these relationships create a sense of belonging and validation that can counteract feelings of isolation and helplessness, and in many cases, the knowledge that one is not alone can be enough to sustain hope during the most trying times; however, resilience is not solely dependent on external support, as there are instances where individuals must rely primarily on their internal resources, and in such cases, qualities such as self-awareness, discipline, and a sense of purpose become particularly important, as they provide a foundation upon which resilience can be built even in the absence of strong external reinforcement; purpose, in particular, acts as a guiding force, giving meaning to struggle and direction to effort, and when individuals have a clear sense of why they are enduring hardship, it becomes easier to tolerate discomfort and remain committed to their goals, as the struggle is no longer seen as pointless but as part of a larger journey; it is also important to recognize that resilience does not mean invulnerability, and resilient individuals are not immune to stress, burnout, or emotional pain, but rather they possess the ability to recover, to regain equilibrium after disruption, and to continue moving forward even when progress is slow or uncertain, and this capacity for recovery is what distinguishes resilience from mere endurance, as it involves not just surviving adversity but adapting to it in a way that preserves or even enhances one’s well-being over time; the role of failure in the development of resilience cannot be overstated, as failure provides critical feedback that, when approached constructively, can lead to significant personal growth, but this requires a mindset that views failure not as a reflection of inherent inadequacy but as a natural part of the learning process, and cultivating such a mindset often involves challenging deeply ingrained beliefs about success and self-worth, which can be difficult but ultimately rewarding, as it opens the door to greater flexibility and openness in the face of challenges; cultural and societal factors also influence how resilience is understood and expressed, as different cultures may emphasize different aspects of coping, whether it is individual perseverance, collective support, spiritual faith, or acceptance of circumstances, and these cultural frameworks shape not only how individuals respond to adversity but also how they interpret their experiences, and recognizing this diversity is important in understanding that there is no single correct way to be resilient, but rather multiple pathways that can lead to similar outcomes of adaptation and growth; education systems and workplaces are increasingly recognizing the importance of resilience, not just as a personal trait but as a skill that can be developed and supported through intentional practices, such as fostering growth mindsets, encouraging reflection, providing constructive feedback, and creating environments that balance challenge with support, and while these efforts are still evolving, they represent a shift towards a more holistic understanding of success, one that values not only achievement but also the ability to navigate setbacks and uncertainties effectively; technology and modern lifestyles have introduced new challenges to resilience, as the pace of life, constant connectivity, and exposure to curated representations of others’ success can create unrealistic expectations and increased pressure, making it more difficult for individuals to cope with their own struggles, and in this context, developing resilience requires not only internal skills but also conscious management of external influences, such as setting boundaries with technology, cultivating mindfulness, and prioritizing authentic experiences over superficial comparisons; the concept of resilience also raises important ethical considerations, particularly in how it is applied in social and organizational contexts, as there is a risk of placing excessive responsibility on individuals to adapt to difficult circumstances without addressing the underlying systemic issues that contribute to those difficulties, and while personal resilience is valuable, it should not be used as a substitute for structural change, and a balanced approach recognizes the importance of both individual capacity and collective responsibility in creating environments that support well-being and growth; ultimately, resilience is not a destination but an ongoing process, one that evolves over time as individuals encounter new experiences and challenges, and it is shaped by a combination of internal traits, external supports, and contextual factors, making it both deeply personal and universally relevant, and by understanding resilience in this nuanced way, it becomes possible to move beyond simplistic notions of toughness or endurance and towards a more compassionate and realistic appreciation of what it means to navigate the complexities of life, and in doing so, individuals can begin to cultivate resilience not as a reaction to hardship alone but as a proactive approach to living, one that embraces uncertainty, values growth, and recognizes the inherent strength that lies within the human capacity to adapt, persist, and find meaning even in the face of adversity.


"""

# ══════════════════════════════════════════════════════════════
#  MAIN — Run both models then evaluate
# ══════════════════════════════════════════════════════════════

gpu_info = get_gpu_info()
print_system_banner(gpu_info)

results = []
for model_cfg in MODELS:
    res = translate_with_model(model_cfg, document_text)
    print_speed_report(res, gpu_info)
    results.append(res)

# Final 3-way accuracy + speed comparison
run_accuracy_evaluation(results, gpu_info)



═════════════════════════════════════════════════════════════════
  🖥️   SYSTEM CONFIGURATION
═════════════════════════════════════════════════════════════════
  Device       : CUDA
  GPU          : Tesla T4
  VRAM         : 15.64 GB
  CUDA         : 12.8
  GPU Count    : 1
  Compute Cap. : 7.5
  Batch Size   : 16  |  Beams: 4
  Models       : NLLB-600M vs mBART-large-50
═════════════════════════════════════════════════════════════════


═════════════════════════════════════════════════════════════════
  🟢  RUNNING : NLLB-600M
      Model   : facebook/nllb-200-distilled-600M
      Lang    : eng_Latn → mar_Deva
═════════════════════════════════════════════════════════════════
⏳ Loading model...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Token indices sequence length is longer than the specified maximum sequence length for this model (1820 > 1024). Running this sequence through the model will result in indexing errors


✅ Model ready in 17.9s

  📄  Parsing document...
   📊 Chunks: 103  |  Min: 11  Max: 194  Avg: 39 tokens
      Over-limit fixed: 1  |  All ≤ 400: ✅

   Words      : 2,774
   Sentences  : 103
   Paragraphs : 21
   GPU Batches: 7

  ⚙️   Translating...
   Batch 1/7  (16 sents)... ✅ 1.35s
   Batch 2/7  (16 sents)... ✅ 1.10s
   Batch 3/7  (16 sents)... ✅ 1.16s
   Batch 4/7  (16 sents)... ✅ 1.26s
   Batch 5/7  (16 sents)... ✅ 1.13s
   Batch 6/7  (16 sents)... ✅ 5.68s
   Batch 7/7  (7 sents)... ✅ 5.23s

🗑️  Unloading NLLB-600M from VRAM...
   ✅ VRAM freed


  ⚡ 🟢 NLLB-600M — Speed
  ─────────────────────────────────────────────────────
  Time Taken        : 16.93s
  Words / Minute    : 9,832.5 WPM
  Words / Second    : 163.9 WPS
  vs 2000 WPM target  : [██████████████████████████████] 491.6%  ✅
  🎉 TARGET MET on single Tesla T4!

═════════════════════════════════════════════════════════════════
  🔵  RUNNING : mBART-large-50
      Model   : facebook/mbart-large-50-many-to-many-mmt
      Lang  

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

✅ Model ready in 10.3s

  📄  Parsing document...
   📊 Chunks: 103  |  Min: 10  Max: 193  Avg: 40 tokens
      Over-limit fixed: 1  |  All ≤ 400: ✅

   Words      : 2,774
   Sentences  : 103
   Paragraphs : 21
   GPU Batches: 7

  ⚙️   Translating...
   Batch 1/7  (16 sents)... ✅ 3.67s
   Batch 2/7  (16 sents)... ✅ 2.74s
   Batch 3/7  (16 sents)... ✅ 3.11s
   Batch 4/7  (16 sents)... ✅ 2.99s
   Batch 5/7  (16 sents)... ✅ 2.04s
   Batch 6/7  (16 sents)... ✅ 14.35s
   Batch 7/7  (7 sents)... ✅ 13.37s

🗑️  Unloading mBART-large-50 from VRAM...
   ✅ VRAM freed


  ⚡ 🔵 mBART-large-50 — Speed
  ─────────────────────────────────────────────────────
  Time Taken        : 42.27s
  Words / Minute    : 3,937.5 WPM
  Words / Second    : 65.6 WPS
  vs 2000 WPM target  : [██████████████████████████████] 196.9%  ✅
  🎉 TARGET MET on single Tesla T4!

  🔬  ACCURACY EVALUATION  (3-way comparison)
  Sentences: 103  |  Models: NLLB-600M vs mBART-large-50

🔴 [1] Google Translate  EN → MR ...
      ✅ Done (1

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


      ✅ Done


═════════════════════════════════════════════════════════════════
  ⚡  SPEED COMPARISON  (vs 2,000 WPM Production Target)
═════════════════════════════════════════════════════════════════

  Model                      WPM     WPS     Time     vs Target
  ──────────────────────────────────────────────────────────
  🟢 NLLB-600M            9,832.5   163.9   16.93s    491.6%  ✅
  🔵 mBART-large-50       3,937.5    65.6   42.27s    196.9%  ✅
  🔴 Google Translate         API       —        —           Ref
  ──────────────────────────────────────────────────────────

  🏆 Fastest Model : 🟢 NLLB-600M (9,832 WPM)

  ── Reference GPU Speeds (NLLB-600M, beam=4) ───────
  GPU                           Est. WPM    ≥2K
  ────────────────────────────  ─────────  ─────
  RTX 3060                           320    
  RTX 3080                           580    
  RTX 3090                           820    
  RTX 4080                         1,050    
  RTX 4090                         1,500   

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


comparsions

In [ ]:
# ══════════════════════════════════════════════════════════════
#  DUAL-MODEL EVALUATOR — Google Colab
#  Tests NLLB-600M vs mBART-large-50 on CSV datasets from Drive
#  Metrics: Standard BLEU, Back-Trans BLEU, Semantic Sim, ChrF
#  Target: 2000 WPM in production
# ══════════════════════════════════════════════════════════════

# ── Step 0: Mount Google Drive ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Step 1: Install dependencies ───────────────────────────────
# !pip install -q sacrebleu sentence-transformers deep-translator transformers

import gc
import re
import time
import warnings
import torch
import numpy as np
import pandas as pd
import sacrebleu
warnings.filterwarnings("ignore")

import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ══════════════════════════════════════════════════════════════
#  CONFIG — Update paths to match your Google Drive structure
# ══════════════════════════════════════════════════════════════

# ── CSV file paths on Google Drive ────────────────────────────
# Columns expected: 'en' (English), 'mr' (Marathi reference)
DRIVE_BASE   = "/content/drive/MyDrive"        # ← change if your files are in a subfolder
TRAIN_CSV    = f"{DRIVE_BASE}/train-eng-mar.csv"
TEST_CSV     = f"{DRIVE_BASE}/test-eng-mar.csv"
VAL_CSV      = f"{DRIVE_BASE}/val-eng-mar.csv"

# ── How many rows to evaluate (keep low to avoid timeout) ─────
EVAL_ROWS    = 200    # rows from test CSV used for accuracy eval
SPEED_TEST_ROWS = 100 # rows from test CSV used for WPM speed test

# ── Model settings ────────────────────────────────────────────
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 16
NUM_BEAMS   = 4
TOKEN_LIMIT = 400

TARGET_WPM  = 2000

# ── Both models + their correct language codes ────────────────
MODELS = [
    {
        "label"    : "NLLB-600M",
        "name"     : "facebook/nllb-200-distilled-600M",
        "src_lang" : "eng_Latn",
        "tgt_lang" : "mar_Deva",
        "emoji"    : "🟢",
    },
    {
        "label"    : "mBART-large-50",
        "name"     : "facebook/mbart-large-50-many-to-many-mmt",
        "src_lang" : "en_XX",     # ← correct code for mBART
        "tgt_lang" : "mr_IN",     # ← correct code for mBART
        "emoji"    : "🔵",
    },
]

# ══════════════════════════════════════════════════════════════
#  STEP 1 — Load CSVs from Google Drive
# ══════════════════════════════════════════════════════════════

print("📂 Loading CSV files from Google Drive...")

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)
df_val   = pd.read_csv(VAL_CSV)

# Normalise column names (handle case/spaces)
for df in [df_train, df_test, df_val]:
    df.columns = [c.strip().lower() for c in df.columns]

print(f"   train : {len(df_train):,} rows  | columns: {list(df_train.columns)}")
print(f"   test  : {len(df_test):,} rows   | columns: {list(df_test.columns)}")
print(f"   val   : {len(df_val):,} rows    | columns: {list(df_val.columns)}")

# ── Sample rows for evaluation (from test set) ────────────────
eval_df  = df_test.dropna(subset=['en', 'mr']).sample(
               n=min(EVAL_ROWS, len(df_test)), random_state=42).reset_index(drop=True)
speed_df = df_test.dropna(subset=['en', 'mr']).sample(
               n=min(SPEED_TEST_ROWS, len(df_test)), random_state=0).reset_index(drop=True)

# Source sentences (EN) and reference translations (MR)
eval_en_sents  = eval_df['en'].tolist()
eval_mr_ref    = eval_df['mr'].tolist()     # ← ground truth Marathi from CSV
speed_en_sents = speed_df['en'].tolist()

print(f"\n   ✅ Eval set   : {len(eval_en_sents)} sentences (from test CSV)")
print(f"   ✅ Speed test : {len(speed_en_sents)} sentences (from test CSV)")

# ══════════════════════════════════════════════════════════════
#  GPU HELPERS
# ══════════════════════════════════════════════════════════════

GPU_REFERENCE_SPEEDS = {
    "RTX 3060"    : 320,   "RTX 3080"    : 580,
    "RTX 3090"    : 820,   "RTX 4080"    : 1050,
    "RTX 4090"    : 1500,  "A10G"        : 1100,
    "A100 (40GB)" : 2200,  "A100 (80GB)" : 2400,
    "H100"        : 4500,  "T4"          : 380,
    "V100 (16GB)" : 700,   "V100 (32GB)" : 750,
    "L4"          : 900,   "CPU (no GPU)": 35,
}


def get_gpu_info() -> dict:
    info = {
        "device": DEVICE.upper(), "gpu_name": "N/A (CPU only)",
        "vram_gb": 0.0, "cuda_version": "N/A",
        "gpu_count": 0, "compute_cap": "N/A",
    }
    if not torch.cuda.is_available():
        return info
    info["gpu_count"]    = torch.cuda.device_count()
    info["gpu_name"]     = torch.cuda.get_device_name(0)
    info["vram_gb"]      = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
    info["cuda_version"] = torch.version.cuda or "N/A"
    cap = torch.cuda.get_device_capability(0)
    info["compute_cap"]  = f"{cap[0]}.{cap[1]}"
    return info


def print_system_banner(gpu: dict, eval_rows: int, speed_rows: int):
    print("\n" + "═"*65)
    print("  🖥️   SYSTEM CONFIGURATION")
    print("═"*65)
    print(f"  Device        : {gpu['device']}")
    print(f"  GPU           : {gpu['gpu_name']}")
    print(f"  VRAM          : {gpu['vram_gb']} GB")
    print(f"  CUDA          : {gpu['cuda_version']}")
    print(f"  Compute Cap.  : {gpu['compute_cap']}")
    print(f"  Batch Size    : {BATCH_SIZE}  |  Beams: {NUM_BEAMS}")
    print(f"  Eval rows     : {eval_rows}  |  Speed test rows: {speed_rows}")
    print(f"  Models        : {' vs '.join(m['label'] for m in MODELS)}")
    print("═"*65 + "\n")


# ══════════════════════════════════════════════════════════════
#  STEP 2 — Batch inference (shared by both models)
# ══════════════════════════════════════════════════════════════

def translate_sentences(sentences: list, model, tokenizer,
                        src_lang: str, tgt_id: int) -> list:
    """Translate a list of sentences in batches. No truncation."""
    tokenizer.src_lang = src_lang
    all_out = []
    total_b = (len(sentences) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in range(0, len(sentences), BATCH_SIZE):
        batch  = sentences[i: i + BATCH_SIZE]
        b_num  = i // BATCH_SIZE + 1
        t_b    = time.time()
        print(f"   Batch {b_num}/{total_b}  ({len(batch)} sents)...", end=" ")

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,        # CSV sentences may vary in length
            max_length=TOKEN_LIMIT,
        ).to(DEVICE)

        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=tgt_id,
                max_length=256,
                num_beams=NUM_BEAMS,
                early_stopping=True,
            )
        decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
        all_out.extend(decoded)
        print(f"✅ {time.time()-t_b:.2f}s")

    return all_out


# ══════════════════════════════════════════════════════════════
#  STEP 3 — Run one model (speed test + accuracy eval)
# ══════════════════════════════════════════════════════════════

def run_model(cfg: dict, speed_sents: list, eval_sents: list) -> dict:
    label    = cfg["label"]
    mname    = cfg["name"]
    src_lang = cfg["src_lang"]
    tgt_lang = cfg["tgt_lang"]
    emoji    = cfg["emoji"]

    print("\n" + "═"*65)
    print(f"  {emoji}  LOADING : {label}")
    print(f"      {mname}  |  {src_lang} → {tgt_lang}")
    print("═"*65)

    # ── Load ──────────────────────────────────────────────────
    t0 = time.time()
    tokenizer = AutoTokenizer.from_pretrained(mname)
    model     = AutoModelForSeq2SeqLM.from_pretrained(
        mname,
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
    ).to(DEVICE)
    model.eval()
    tokenizer.src_lang = src_lang
    tgt_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    print(f"✅ Model loaded in {time.time()-t0:.1f}s\n")

    # ── Speed test ────────────────────────────────────────────
    print("=" * 65)
    print(f"  ⚡  SPEED TEST  ({len(speed_sents)} sentences from test CSV)")
    print("=" * 65)
    n_words_speed = sum(len(s.split()) for s in speed_sents)
    t_speed = time.time()
    _ = translate_sentences(speed_sents, model, tokenizer, src_lang, tgt_id)
    t_speed = time.time() - t_speed
    wpm  = round(n_words_speed / t_speed * 60, 1)
    wps  = round(n_words_speed / t_speed, 1)
    print(f"\n   Words : {n_words_speed:,}  |  Time : {t_speed:.2f}s")
    print(f"   WPM   : {wpm:,.1f}  |  WPS  : {wps:.1f}")
    pct  = round(wpm / TARGET_WPM * 100, 1)
    icon = "✅" if pct >= 100 else "❌"
    print(f"   vs {TARGET_WPM} WPM target : {pct}%  {icon}\n")

    # ── Accuracy eval ─────────────────────────────────────────
    print("=" * 65)
    print(f"  🔬  ACCURACY EVAL  ({len(eval_sents)} sentences from test CSV)")
    print("=" * 65)
    t_eval = time.time()
    translations = translate_sentences(eval_sents, model, tokenizer, src_lang, tgt_id)
    t_eval = time.time() - t_eval
    print(f"\n   ✅ Translated {len(translations)} sentences in {t_eval:.2f}s\n")

    # ── Unload VRAM ───────────────────────────────────────────
    print(f"🗑️  Unloading {label} from VRAM...")
    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    gc.collect()
    print("   ✅ VRAM freed\n")

    return {
        "label"        : label,
        "emoji"        : emoji,
        "translations" : translations,          # model MR output
        "stats": {
            "n_words"  : n_words_speed,
            "time_s"   : round(t_speed, 2),
            "wpm"      : wpm,
            "wps"      : wps,
            "pct"      : pct,
        },
    }


# ══════════════════════════════════════════════════════════════
#  STEP 4 — Compute all metrics (Standard BLEU + Back-Trans + Sim)
# ══════════════════════════════════════════════════════════════

def compute_metrics(results: list, en_sents: list, mr_ref: list, gpu: dict):
    n = len(en_sents)
    print("\n" + "="*65)
    print("  📐  COMPUTING METRICS")
    print("="*65)

    # ── Google Translate (EN → MR) ────────────────────────────
    print("\n🔴 Google Translate  EN → MR ...")
    gt_en = GoogleTranslator(source="en", target="mr")
    mr_google = []
    for s in en_sents:
        try:    mr_google.append(gt_en.translate(s) or "[empty]")
        except: mr_google.append("[error]")
        time.sleep(0.15)
    print(f"   ✅ Done ({len(mr_google)} sentences)")

    # ── Back-translate all outputs MR → EN ────────────────────
    gt_mr = GoogleTranslator(source="mr", target="en")

    def back_translate(sents, tag):
        print(f"   Back-translating {tag} (MR → EN)...")
        bt = []
        for s in sents:
            try:    bt.append(gt_mr.translate(s) or "[empty]")
            except: bt.append("[error]")
            time.sleep(0.15)
        return bt

    bt_google = back_translate(mr_google, "Google")
    for r in results:
        r['bt'] = back_translate(r['translations'], r['label'])

    # ── Semantic similarity (EN vs MR) ────────────────────────
    print("\n🧠 Computing multilingual semantic similarity...")
    emb_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
    en_embs   = emb_model.encode(en_sents, show_progress_bar=False)
    g_embs    = emb_model.encode(mr_google, show_progress_bar=False)
    ref_embs  = emb_model.encode(mr_ref, show_progress_bar=False)
    g_sims    = [float(cosine_similarity([e],[g])[0][0])
                 for e, g in zip(en_embs, g_embs)]

    for r in results:
        m_embs     = emb_model.encode(r['translations'], show_progress_bar=False)
        r['sims']  = [float(cosine_similarity([e],[m])[0][0])
                      for e, m in zip(en_embs, m_embs)]
    print("   ✅ Done\n")

    # ── Compute all scores ─────────────────────────────────────
    # Google scores (vs CSV reference MR)
    g_std_bleu  = sacrebleu.corpus_bleu(mr_google,  [mr_ref]).score     # standard BLEU vs ref
    g_bt_bleu   = sacrebleu.corpus_bleu(bt_google,  [en_sents]).score   # back-trans BLEU
    g_chrf      = sacrebleu.corpus_chrf(mr_google,  [mr_ref]).score     # ChrF vs ref
    g_avg_sim   = float(np.mean(g_sims))

    for r in results:
        r['std_bleu'] = sacrebleu.corpus_bleu(r['translations'], [mr_ref]).score      # ← standard BLEU vs reference
        r['bt_bleu']  = sacrebleu.corpus_bleu(r['bt'],           [en_sents]).score    # ← back-translation BLEU
        r['chrf']     = sacrebleu.corpus_chrf(r['translations'],  [mr_ref]).score      # ← ChrF vs reference
        r['avg_sim']  = float(np.mean(r['sims']))
        r['wins_vs_google'] = sum(s > g for s, g in zip(r['sims'], g_sims))

    # ══════════════════════════════════════════════════════════
    #  SPEED TABLE
    # ══════════════════════════════════════════════════════════
    print("\n" + "═"*65)
    print(f"  ⚡  SPEED COMPARISON  (target: {TARGET_WPM:,} WPM)")
    print("═"*65)
    print(f"\n  {'Model':<22} {'WPM':>9}  {'WPS':>6}  {'Time':>7}  {'Target %':>9}")
    print("  " + "─"*58)
    for r in results:
        s = r['stats']
        icon = "✅" if s['pct'] >= 100 else "❌"
        print(f"  {r['emoji']} {r['label']:<20} {s['wpm']:>9,.1f}"
              f"  {s['wps']:>6,.1f}  {s['time_s']:>6.2f}s  {s['pct']:>7.1f}%  {icon}")
    print(f"  🔴 {'Google Translate':<20} {'API':>9}  {'—':>6}  {'—':>7}  {'Ref':>9}")
    print("  " + "─"*58)
    fastest = max(results, key=lambda r: r['stats']['wpm'])
    print(f"\n  🏆 Fastest : {fastest['emoji']} {fastest['label']} "
          f"({fastest['stats']['wpm']:,.0f} WPM)\n")

    # GPU Reference table
    print(f"  ── GPU Reference Speeds ────────────────────────────")
    print(f"  {'GPU':<28} {'Est. WPM':>9}  {'≥ 2K':>5}")
    print(f"  {'─'*28}  {'─'*9}  {'─'*5}")
    for gname, wpmref in GPU_REFERENCE_SPEEDS.items():
        meets  = "✅" if wpmref >= TARGET_WPM else "  "
        marker = " ◄ YOU" if gname in gpu["gpu_name"] else ""
        print(f"  {gname:<28} {wpmref:>9,}  {meets}{marker}")

    # ══════════════════════════════════════════════════════════
    #  ACCURACY TABLE
    # ══════════════════════════════════════════════════════════
    print("\n" + "═"*65)
    print("  🏆  ACCURACY SCORECARD")
    print(f"      Eval set: {n} sentences from test-eng-mar.csv")
    print(f"      Reference: CSV 'mr' column (ground-truth Marathi)")
    print("═"*65)

    col = 16
    hdr = (f"\n  {'Metric':<34}"
           f"{'🔴 Google':>{col}}"
           + "".join(f"{r['emoji']+' '+r['label']:>{col}}" for r in results))
    print(hdr)
    print("  " + "─"*72)

    def _row(name, g_val, model_vals, fmt=".3f"):
        vals = [g_val] + model_vals
        best = max(vals)
        cells = []
        for v in vals:
            star = "★" if abs(v - best) < 1e-9 else " "
            cells.append(f"{v:{col-2}{fmt}} {star}")
        return f"  {name:<34}" + "".join(cells)

    print(_row("Standard BLEU  vs CSV ref  (↑)",
               g_std_bleu, [r['std_bleu'] for r in results]))
    print(_row("Back-Trans BLEU vs EN src   (↑)",
               g_bt_bleu,  [r['bt_bleu'] for r in results]))
    print(_row("ChrF  vs CSV ref            (↑)",
               g_chrf,     [r['chrf']    for r in results]))
    print(_row("Avg Semantic Sim (EN vs MR) (↑)",
               g_avg_sim,  [r['avg_sim'] for r in results]))

    print("  " + "─"*72)
    print(f"\n  Sentence wins (sim vs Google):")
    for r in results:
        print(f"    {r['emoji']} {r['label']:<18} : {r['wins_vs_google']}/{n} sentences")

    # Head-to-head
    if len(results) == 2:
        r1, r2 = results
        h2h = sum(a > b for a, b in zip(r1['sims'], r2['sims']))
        print(f"\n  Head-to-head (semantic sim):")
        print(f"    {r1['emoji']} {r1['label']} : {h2h}/{n}  vs  "
              f"{r2['emoji']} {r2['label']} : {n-h2h}/{n}")

    # Overall winners
    def composite(g_val_list, model_val_list):
        return float(np.mean(model_val_list))

    def total_score(r):
        return r['std_bleu'] + r['bt_bleu'] + r['chrf'] + r['avg_sim'] * 100

    q_winner = max(results, key=total_score)
    s_winner = fastest
    print(f"\n  🏆 Quality Winner : {q_winner['emoji']} {q_winner['label']}")
    print(f"  ⚡ Speed Winner   : {s_winner['emoji']} {s_winner['label']}")

    # ══════════════════════════════════════════════════════════
    #  SAMPLE SENTENCES
    # ══════════════════════════════════════════════════════════
    print("\n" + "═"*65)
    print("  📋  SAMPLE SENTENCES (first 5 from eval set)")
    print("═"*65 + "\n")

    for i in range(min(5, n)):
        en   = en_sents[i]
        ref  = mr_ref[i]
        g_mr = mr_google[i]
        g_s  = g_sims[i]
        print(f"  [{i+1:02d}] EN  : {en}")
        print(f"        REF : {ref}  ← CSV ground truth")
        for r in results:
            mr  = r['translations'][i]
            bt  = r['bt'][i]
            s   = r['sims'][i]
            print(f"        {r['emoji']} {r['label']:<16}: {mr}")
            print(f"             back-EN : {bt}  [sim={s:.3f}]")
        print(f"        🔴 Google          : {g_mr}  [sim={g_s:.3f}]")
        print()

    # ══════════════════════════════════════════════════════════
    #  METRIC LEGEND
    # ══════════════════════════════════════════════════════════
    print("  ℹ️  Standard BLEU  : model Marathi output vs CSV reference Marathi")
    print("  ℹ️  Back-Trans BLEU: model MR back-translated to EN vs original EN")
    print("  ℹ️  ChrF           : character n-gram overlap vs CSV reference")
    print("  ℹ️  Semantic Sim   : multilingual embedding cosine sim (EN vs MR)")
    print("  ℹ️  ★              : best score for that metric\n")

    return {
        "google": {
            "std_bleu": g_std_bleu, "bt_bleu": g_bt_bleu,
            "chrf": g_chrf, "avg_sim": g_avg_sim,
        }
    }


# ══════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════

gpu_info = get_gpu_info()
print_system_banner(gpu_info, len(eval_en_sents), len(speed_en_sents))

# ── Run both models sequentially (free VRAM between each) ─────
all_results = []
for cfg in MODELS:
    res = run_model(cfg, speed_en_sents, eval_en_sents)
    all_results.append(res)

# ── Final 3-way comparison ────────────────────────────────────
compute_metrics(all_results, eval_en_sents, eval_mr_ref, gpu_info)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Loading CSV files from Google Drive...
   train : 100,000 rows  | columns: ['en', 'mr']
   test  : 20,000 rows   | columns: ['en', 'mr']
   val   : 10,000 rows    | columns: ['en', 'mr']

   ✅ Eval set   : 200 sentences (from test CSV)
   ✅ Speed test : 100 sentences (from test CSV)

═════════════════════════════════════════════════════════════════
  🖥️   SYSTEM CONFIGURATION
═════════════════════════════════════════════════════════════════
  Device        : CUDA
  GPU           : Tesla T4
  VRAM          : 15.64 GB
  CUDA          : 12.8
  Compute Cap.  : 7.5
  Batch Size    : 16  |  Beams: 4
  Eval rows     : 200  |  Speed test rows: 100
  Models        : NLLB-600M vs mBART-large-50
═════════════════════════════════════════════════════════════════


═════════════════════════════════════════════════════════════════
  🟢  LOADING : NLLB-600M
      facebook/n

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✅ Done


═════════════════════════════════════════════════════════════════
  ⚡  SPEED COMPARISON  (target: 2,000 WPM)
═════════════════════════════════════════════════════════════════

  Model                        WPM     WPS     Time   Target %
  ──────────────────────────────────────────────────────────
  🟢 NLLB-600M              3,296.2    54.9   22.52s    164.8%  ✅
  🔵 mBART-large-50         2,796.5    46.6   26.54s    139.8%  ✅
  🔴 Google Translate           API       —        —        Ref
  ──────────────────────────────────────────────────────────

  🏆 Fastest : 🟢 NLLB-600M (3,296 WPM)

  ── GPU Reference Speeds ────────────────────────────
  GPU                           Est. WPM   ≥ 2K
  ────────────────────────────  ─────────  ─────
  RTX 3060                           320    
  RTX 3080                           580    
  RTX 3090                           820    
  RTX 4080                         1,050    
  RTX 4090                         1,500    
  A10G           

{'google': {'std_bleu': 12.856177316707923,
  'bt_bleu': 50.562719479338405,
  'chrf': 42.49089492607708,
  'avg_sim': 0.823905820697546}}

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  PDF DUAL-MODEL TRANSLATOR — Google Colab (.ipynb)          ║
# ║  Upload a PDF → NLLB-600M vs mBART-large-50 comparison      ║
# ║  No ngrok. No server. Just run this cell.                   ║
# ╚══════════════════════════════════════════════════════════════╝

# ── 0. Install ─────────────────────────────────────────────────
import subprocess, sys
for pkg in ["pdfplumber", "sacrebleu", "sentence-transformers",
            "deep-translator", "transformers"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# ── 1. Imports ─────────────────────────────────────────────────
import gc, io, re, time, warnings
import torch, numpy as np, sacrebleu, pdfplumber
warnings.filterwarnings("ignore")

from google.colab import files as colab_files
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from deep_translator import GoogleTranslator

try:
    import nltk; nltk.download("punkt_tab", quiet=True)
    from nltk.tokenize import sent_tokenize
    HAS_NLTK = True
except Exception:
    HAS_NLTK = False

# ── 2. Config ──────────────────────────────────────────────────
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 16
NUM_BEAMS   = 4
TOKEN_LIMIT = 400
TARGET_WPM  = 2000

MODELS_CFG = [
    {"label": "NLLB-600M",      "emoji": "🟢",
     "name": "facebook/nllb-200-distilled-600M",
     "src_lang": "eng_Latn", "tgt_lang": "mar_Deva"},
    {"label": "mBART-large-50", "emoji": "🔵",
     "name": "facebook/mbart-large-50-many-to-many-mmt",
     "src_lang": "en_XX",    "tgt_lang": "mr_IN"},
]

# ── 3. Helpers ─────────────────────────────────────────────────
def pdf_to_text(raw_bytes):
    parts = []
    with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t: parts.append(t.strip())
    return "\n\n".join(parts)

SPLIT_PATS = [r'(?<=\.\s)', r'(?<=;\s)', r'(?<=,\s)',
              r'\s+(?=and|but|or|however|therefore|moreover)\s+', r'\s+']

def _ntok(tok, src_lang, text):
    tok.src_lang = src_lang
    return len(tok(text, add_special_tokens=True)["input_ids"])

def _sub_split(tok, src_lang, sent, limit):
    if _ntok(tok, src_lang, sent) <= limit: return [sent]
    for pat in SPLIT_PATS:
        parts = [p.strip() for p in re.split(pat, sent) if p.strip()]
        if len(parts) > 1:
            out = []
            for p in parts:
                out.extend(_sub_split(tok, src_lang, p, limit)
                           if _ntok(tok, src_lang, p) > limit else [p])
            return out
    w = sent.split(); mid = len(w)//2
    return (_sub_split(tok, src_lang, " ".join(w[:mid]), limit) +
            _sub_split(tok, src_lang, " ".join(w[mid:]), limit))

def parse_text(text, tok, src_lang):
    items = []
    for i, para in enumerate(re.split(r'\n\s*\n', text.strip())):
        if i > 0: items.append({"type": "br"})
        para = para.strip()
        if not para: continue
        raw = (sent_tokenize(para, language="english") if HAS_NLTK
               else re.split(r'(?<=[.!?])\s+', para))
        for s in raw:
            s = s.strip()
            if not s: continue
            chunks = ([s] if _ntok(tok, src_lang, s) <= TOKEN_LIMIT
                      else _sub_split(tok, src_lang, s, TOKEN_LIMIT))
            for c in chunks:
                items.append({"type": "sent", "text": c})
    sents = [it["text"] for it in items if it["type"] == "sent"]
    return sents, items

def translate_all(sents, model, tok, src_lang, tgt_id):
    all_tr, total = [], (len(sents) + BATCH_SIZE - 1) // BATCH_SIZE
    for i in range(0, len(sents), BATCH_SIZE):
        batch = sents[i: i + BATCH_SIZE]
        b = i // BATCH_SIZE + 1
        print(f"   Batch {b}/{total} ({len(batch)} sents)...", end=" ", flush=True)
        tok.src_lang = src_lang
        inp = tok(batch, return_tensors="pt", padding=True,
                  truncation=True, max_length=TOKEN_LIMIT).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inp, forced_bos_token_id=tgt_id,
                                 max_length=512, num_beams=NUM_BEAMS,
                                 early_stopping=True)
        decoded = tok.batch_decode(out, skip_special_tokens=True)
        all_tr.extend(decoded)
        print(f"✅")
    return all_tr

def reconstruct(items, translations):
    it, parts, cur = iter(translations), [], []
    for x in items:
        if x["type"] == "br":
            if cur: parts.append(" ".join(cur)); cur = []
            parts.append("\n\n")
        else: cur.append(next(it))
    if cur: parts.append(" ".join(cur))
    return "".join(parts).strip()

def print_separator(title=""):
    print("\n" + "═"*65)
    if title: print(f"  {title}"); print("═"*65)

# ── 4. Upload PDF ──────────────────────────────────────────────
print_separator("📄  Upload your PDF")
print("  A file picker will appear below ↓")
uploaded = colab_files.upload()

filename = list(uploaded.keys())[0]
raw_bytes = uploaded[filename]
text = pdf_to_text(raw_bytes)
n_words = len(text.split())

print(f"\n  ✅ File     : {filename}")
print(f"  📊 Words   : {n_words:,}")
print(f"  📝 Preview : {text[:200].replace(chr(10),' ')}…")

if n_words < 10:
    raise ValueError("❌ Could not extract text. Please use a text-based (not scanned) PDF.")

# ── 5. System banner ──────────────────────────────────────────
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram     = (round(torch.cuda.get_device_properties(0).total_memory/1e9, 2)
            if torch.cuda.is_available() else 0)
print_separator("🖥️   SYSTEM")
print(f"  GPU        : {gpu_name}  ({vram} GB)")
print(f"  Device     : {DEVICE.upper()}")
print(f"  Batch/Beams: {BATCH_SIZE} / {NUM_BEAMS}")
print(f"  Target WPM : {TARGET_WPM:,}")

# ── 6. Run both models ─────────────────────────────────────────
results = []

for cfg in MODELS_CFG:
    print_separator(f"{cfg['emoji']}  {cfg['label']}")
    print(f"  Loading model …")
    t0  = time.time()
    tok = AutoTokenizer.from_pretrained(cfg["name"])
    mdl = AutoModelForSeq2SeqLM.from_pretrained(
        cfg["name"],
        dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        low_cpu_mem_usage=True,
    ).to(DEVICE)
    mdl.eval()
    tok.src_lang  = cfg["src_lang"]
    tgt_id = tok.convert_tokens_to_ids(cfg["tgt_lang"])
    print(f"  ✅ Loaded in {time.time()-t0:.1f}s\n")

    sents, items = parse_text(text, tok, cfg["src_lang"])
    n_sents = len(sents)
    print(f"  Sentences  : {n_sents}  |  "
          f"Batches: {(n_sents + BATCH_SIZE - 1)//BATCH_SIZE}\n")

    t_start = time.time()
    translations = translate_all(sents, mdl, tok, cfg["src_lang"], tgt_id)
    t_total = time.time() - t_start

    wpm = round(n_words / t_total * 60, 1)
    pct = round(wpm / TARGET_WPM * 100, 1)
    bar = "█" * min(30, int(30 * wpm / TARGET_WPM)) + \
          "░" * max(0, 30 - int(30 * wpm / TARGET_WPM))
    icon = "✅" if pct >= 100 else "❌"

    print(f"\n  WPM   : {wpm:,.1f}  ({pct}% of target)  {icon}")
    print(f"  Time  : {t_total:.2f}s  |  {n_words:,} words")
    print(f"  [{bar}]")

    full_translation = reconstruct(items, translations)

    # Unload from VRAM
    del mdl, tok
    if DEVICE == "cuda": torch.cuda.empty_cache()
    gc.collect()
    print(f"\n  🗑️  VRAM freed")

    results.append({
        "label": cfg["label"], "emoji": cfg["emoji"],
        "wpm": wpm, "time_s": round(t_total, 2),
        "pct": pct, "n_sents": n_sents,
        "sentences_en": sents,
        "sentences_mr": translations,
        "full_translation": full_translation,
    })

# ── 7. Accuracy metrics ────────────────────────────────────────
print_separator("🔬  ACCURACY EVALUATION  (Google Translate as reference)")
en_sents = results[0]["sentences_en"]
n = min(len(en_sents), 100)   # cap at 100 to avoid API timeout
en_eval  = en_sents[:n]

print(f"  Evaluating first {n} sentences via Google Translate …\n")

gt_en = GoogleTranslator(source="en", target="mr")
gt_mr = GoogleTranslator(source="mr", target="en")

print("  🔴 Google EN→MR …", end=" ", flush=True)
mr_google = []
for s in en_eval:
    try:    mr_google.append(gt_en.translate(s) or "")
    except: mr_google.append("")
    time.sleep(0.12)
print("✅")

print("  🔴 Google back MR→EN …", end=" ", flush=True)
bt_google = []
for s in mr_google:
    try:    bt_google.append(gt_mr.translate(s) or "")
    except: bt_google.append("")
    time.sleep(0.12)
print("✅")

for r in results:
    mr_eval = r["sentences_mr"][:n]
    print(f"  {r['emoji']} Back-trans {r['label']} MR→EN …", end=" ", flush=True)
    r["bt"] = []
    for s in mr_eval:
        try:    r["bt"].append(gt_mr.translate(s) or "")
        except: r["bt"].append("")
        time.sleep(0.12)
    print("✅")

print("\n  🧠 Semantic similarity …", end=" ", flush=True)
emb_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
en_embs   = emb_model.encode(en_eval)
g_embs    = emb_model.encode(mr_google)
g_sims    = [float(cosine_similarity([e],[g])[0][0]) for e,g in zip(en_embs, g_embs)]

for r in results:
    m_embs     = emb_model.encode(r["sentences_mr"][:n])
    r["sims"]  = [float(cosine_similarity([e],[m])[0][0]) for e,m in zip(en_embs, m_embs)]
    r["avg_sim"]  = round(float(np.mean(r["sims"])), 4)
    r["bt_bleu"]  = round(sacrebleu.corpus_bleu(r["bt"],             [en_eval]).score, 2)
    r["chrf"]     = round(sacrebleu.corpus_chrf(r["sentences_mr"][:n], [mr_google]).score, 2)
    r["wins"]     = sum(s > g for s, g in zip(r["sims"], g_sims))
print("✅")

g_avg_sim = round(float(np.mean(g_sims)), 4)
g_bt_bleu = round(sacrebleu.corpus_bleu(bt_google, [en_eval]).score, 2)

# ── 8. Print scorecard ────────────────────────────────────────
print_separator("⚡  SPEED COMPARISON")
print(f"\n  {'Model':<22} {'WPM':>9}  {'Time':>7}  {'vs Target':>10}")
print("  " + "─"*54)
for r in results:
    icon = "✅" if r["pct"] >= 100 else "❌"
    print(f"  {r['emoji']} {r['label']:<20} {r['wpm']:>9,.1f}  "
          f"{r['time_s']:>6.2f}s  {r['pct']:>7.1f}%  {icon}")
print(f"  🔴 {'Google Translate':<20} {'API':>9}  {'—':>7}  {'Ref':>10}")

print_separator("🏆  ACCURACY SCORECARD")
col = 16
print(f"\n  {'Metric':<34}"
      f"{'🔴 Google':>{col}}"
      + "".join(f"{r['emoji']+' '+r['label']:>{col}}" for r in results))
print("  " + "─"*72)

def row(name, g_val, vals, fmt=".3f"):
    all_v = [g_val] + vals
    best  = max(all_v)
    cells = [f"{v:{col-2}{fmt}} {'★' if abs(v-best)<1e-9 else ' '}" for v in all_v]
    return f"  {name:<34}" + "".join(cells)

print(row("Back-Trans BLEU (↑)", g_bt_bleu,  [r["bt_bleu"]  for r in results]))
print(row("Avg Semantic Sim (↑)", g_avg_sim,  [r["avg_sim"]  for r in results]))
print(row("ChrF vs Google (↑)",  100.0,       [r["chrf"]     for r in results]))

print("\n  Sentence wins vs Google (semantic sim):")
for r in results:
    print(f"    {r['emoji']} {r['label']:<18} : {r['wins']}/{n}")

if len(results) == 2:
    h = sum(a > b for a, b in zip(results[0]["sims"], results[1]["sims"]))
    print(f"\n  Head-to-head  🟢 NLLB: {h}/{n}  vs  🔵 mBART: {n-h}/{n}")

# Overall winners
speed_winner   = max(results, key=lambda r: r["wpm"])
quality_winner = max(results, key=lambda r: r["bt_bleu"] + r["avg_sim"]*100 + r["chrf"])
print(f"\n  ⚡ Speed Winner   : {speed_winner['emoji']}  {speed_winner['label']}  ({speed_winner['wpm']:,.0f} WPM)")
print(f"  🏆 Quality Winner : {quality_winner['emoji']}  {quality_winner['label']}")

# ── 9. Sample sentences ───────────────────────────────────────
print_separator("📋  SAMPLE SENTENCES (first 5)")
for i in range(min(5, n)):
    print(f"\n  [{i+1:02d}] {en_eval[i]}")
    for r in results:
        sim = r["sims"][i]
        print(f"      {r['emoji']} {r['label']:<16}: {r['sentences_mr'][i]}")
        print(f"           back-EN : {r['bt'][i]}  [sim={sim:.3f}]")
    print(f"      🔴 Google          : {mr_google[i]}  [sim={g_sims[i]:.3f}]")

# ── 10. Print full translations ───────────────────────────────
for r in results:
    print_separator(f"{r['emoji']}  {r['label']} — Full Marathi Translation")
    print()
    print(r["full_translation"])

print_separator("✅  DONE")
print(f"  File      : {filename}")
print(f"  Words     : {n_words:,}")
for r in results:
    print(f"  {r['emoji']} {r['label']:<18}: {r['wpm']:,.0f} WPM  |  "
          f"BT-BLEU {r['bt_bleu']}  |  Sim {r['avg_sim']}")



═════════════════════════════════════════════════════════════════
  📄  Upload your PDF
═════════════════════════════════════════════════════════════════
  A file picker will appear below ↓


Saving attention all you need.pdf to attention all you need.pdf

  ✅ File     : attention all you need.pdf
  📊 Words   : 2,033
  📝 Preview : Providedproperattributionisprovided,Googleherebygrantspermissionto reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor scholarlyworks. Attention Is All You Need AshishVaswani∗ NoamShaz…

═════════════════════════════════════════════════════════════════
  🖥️   SYSTEM
═════════════════════════════════════════════════════════════════
  GPU        : Tesla T4  (15.64 GB)
  Device     : CUDA
  Batch/Beams: 16 / 4
  Target WPM : 2,000

═════════════════════════════════════════════════════════════════
  🟢  NLLB-600M
═════════════════════════════════════════════════════════════════
  Loading model …


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


  ✅ Loaded in 16.1s

  Sentences  : 349  |  Batches: 22

   Batch 1/22 (16 sents)... ✅
   Batch 2/22 (16 sents)... ✅
   Batch 3/22 (16 sents)... ✅
   Batch 4/22 (16 sents)... ✅
   Batch 5/22 (16 sents)... ✅
   Batch 6/22 (16 sents)... ✅
   Batch 7/22 (16 sents)... ✅
   Batch 8/22 (16 sents)... ✅
   Batch 9/22 (16 sents)... ✅
   Batch 10/22 (16 sents)... ✅
   Batch 11/22 (16 sents)... ✅
   Batch 12/22 (16 sents)... ✅
   Batch 13/22 (16 sents)... ✅
   Batch 14/22 (16 sents)... ✅
   Batch 15/22 (16 sents)... ✅
   Batch 16/22 (16 sents)... ✅
   Batch 17/22 (16 sents)... ✅
   Batch 18/22 (16 sents)... ✅
   Batch 19/22 (16 sents)... ✅
   Batch 20/22 (16 sents)... ✅
   Batch 21/22 (16 sents)... ✅
   Batch 22/22 (13 sents)... ✅

  WPM   : 473.5  (23.7% of target)  ❌
  Time  : 257.63s  |  2,033 words
  [███████░░░░░░░░░░░░░░░░░░░░░░░]

  🗑️  VRAM freed

═════════════════════════════════════════════════════════════════
  🔵  mBART-large-50
═════════════════════════════════════════════════════════

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

  ✅ Loaded in 13.1s

  Sentences  : 349  |  Batches: 22

   Batch 1/22 (16 sents)... ✅
   Batch 2/22 (16 sents)... ✅
   Batch 3/22 (16 sents)... ✅
   Batch 4/22 (16 sents)... ✅
   Batch 5/22 (16 sents)... ✅
   Batch 6/22 (16 sents)... ✅
   Batch 7/22 (16 sents)... ✅
   Batch 8/22 (16 sents)... ✅
   Batch 9/22 (16 sents)... ✅
   Batch 10/22 (16 sents)... ✅
   Batch 11/22 (16 sents)... ✅
   Batch 12/22 (16 sents)... ✅
   Batch 13/22 (16 sents)... ✅
   Batch 14/22 (16 sents)... ✅
   Batch 15/22 (16 sents)... ✅
   Batch 16/22 (16 sents)... ✅
   Batch 17/22 (16 sents)... ✅
   Batch 18/22 (16 sents)... ✅
   Batch 19/22 (16 sents)... ✅
   Batch 20/22 (16 sents)... ✅
   Batch 21/22 (16 sents)... ✅
   Batch 22/22 (13 sents)... ✅

  WPM   : 493.1  (24.7% of target)  ❌
  Time  : 247.39s  |  2,033 words
  [███████░░░░░░░░░░░░░░░░░░░░░░░]

  🗑️  VRAM freed

═════════════════════════════════════════════════════════════════
  🔬  ACCURACY EVALUATION  (Google Translate as reference)
═══════════════════

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅

═════════════════════════════════════════════════════════════════
  ⚡  SPEED COMPARISON
═════════════════════════════════════════════════════════════════

  Model                        WPM     Time   vs Target
  ──────────────────────────────────────────────────────
  🟢 NLLB-600M                473.5  257.63s     23.7%  ❌
  🔵 mBART-large-50           493.1  247.39s     24.7%  ❌
  🔴 Google Translate           API        —         Ref

═════════════════════════════════════════════════════════════════
  🏆  ACCURACY SCORECARD
═════════════════════════════════════════════════════════════════

  Metric                                    🔴 Google     🟢 NLLB-600M🔵 mBART-large-50
  ────────────────────────────────────────────────────────────────────────
  Back-Trans BLEU (↑)                       25.360          26.370 ★        17.510  
  Avg Semantic Sim (↑)                       0.688           0.896 ★         0.529  
  ChrF vs Google (↑)                       100.000 ★        25.100     

In [ ]:
import pandas as pd
from datasets import load_dataset
from google.colab import drive

# ── 1. MOUNT DRIVE ───────────────────────────────────────────
drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/'  # Change if you want a specific folder

# ── 2. LOAD DATASET (Shuffled) ───────────────────────────────
print("⏳ Loading dataset from Hugging Face...")
# We use split='train' to get the main data, then shuffle it
ds = load_dataset("Hemanth-thunder/english-to-marathi-mt", split='train').shuffle(seed=42)

# Check columns (usually it's 'english' and 'marathi')
print(f"✅ Dataset features: {ds.column_names}")

# ── 3. EXTRACT SLICES (Lakhs) ───────────────────────────────
# Define counts
train_count = 500_000   # 5 Lakh
test_count  = 200_000   # 2 Lakh
val_count   = 100_000   # 1 Lakh

print(f"🚀 Slicing data (Total: {train_count + test_count + val_count} rows)...")

# Slice indices
train_data = ds.select(range(0, train_count))
test_data  = ds.select(range(train_count, train_count + test_count))
val_data   = ds.select(range(train_count + test_count, train_count + test_count + val_count))

# ── 4. CONVERT & SAVE AS CSV ────────────────────────────────
def save_split_to_csv(dataset, filename):
    filepath = SAVE_PATH + filename
    print(f"📦 Converting {filename} to CSV...")
    # Using pandas to save as CSV efficiently
    df = dataset.to_pandas()
    df.to_csv(filepath, index=False)
    print(f"✅ Saved to: {filepath}")

# Save Train
save_split_to_csv(train_data, 'train-eng-mar.csv')
# Save Test
save_split_to_csv(test_data, 'test-eng-mar.csv')
# Save Val
save_split_to_csv(val_data, 'val-eng-mar.csv')

print("\n🎉 ALL DATASETS CREATED SUCCESSFULLY!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Loading dataset from Hugging Face...


README.md:   0%|          | 0.00/404 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/235M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/235M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3627480 [00:00<?, ? examples/s]

✅ Dataset features: ['en', 'mr']
🚀 Slicing data (Total: 800000 rows)...
📦 Converting train-eng-mar.csv to CSV...
✅ Saved to: /content/drive/MyDrive/train-eng-mar.csv
📦 Converting test-eng-mar.csv to CSV...
✅ Saved to: /content/drive/MyDrive/test-eng-mar.csv
📦 Converting val-eng-mar.csv to CSV...
✅ Saved to: /content/drive/MyDrive/val-eng-mar.csv

🎉 ALL DATASETS CREATED SUCCESSFULLY!
